# NPOS on CIFAR-100 — Repo-Faithful Reproduction

This notebook starts a faithful reproduction of the **training-from-scratch CIFAR-100** experiment from **Non-Parametric Outlier Synthesis (NPOS), ICLR 2023**.

Official repository: `deeplearning-wisc/npos`

This first stage implements only the foundation:

1. environment and reproducibility
2. CIFAR-100 configuration
3. repo-faithful two-view CIFAR-100 augmentation
4. CIFAR-style ResNet-34
5. the NPOS model wrapper
6. tensor-shape verification

We deliberately **do not implement NPOS synthesis yet**. First we verify that the data and model match the repository structure.

### Source files used

- `training_from_scratch/cifar.py`
- `training_from_scratch/models/resnet_outliers.py`
- `training_from_scratch/train_CIFAR100.py`
- `training_from_scratch/train_npos_cifar100.sh`


## 1. Environment and reproducibility

The original repository was written around CUDA. This notebook detects CUDA, Apple MPS, or CPU so the data/model foundation can still be inspected on different machines.

Later, when we add the official FAISS-based NPOS synthesis, CUDA + FAISS GPU is the closest match to the authors' implementation.


In [1]:
!pip install -q faiss-cpu
import random
import sys
from dataclasses import dataclass

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms

SEED = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


Python: 3.12.13
PyTorch: 2.10.0+cu128
TorchVision: 0.25.0+cu128
Device: cuda
CUDA device: Tesla T4


## 2. CIFAR-100 / NPOS configuration

The paper's training-from-scratch table and the repository use:

- CIFAR-100 as ID
- CIFAR-style ResNet-34
- 512-D penultimate ResNet feature
- 128-D projected representation
- 500 epochs
- batch size 256
- initial learning rate 0.5
- momentum 0.9
- weight decay $10^{-4}$
- NPOS begins at epoch 200
- class queue size 600
- $k=300$
- 200 boundary candidates
- perturbation scale 0.1
- temperature 0.1

### Important repo discrepancy to resolve later

`train_CIFAR100.py` defines `ID_points_num=2` by default, but `train_npos_cifar100.sh` explicitly passes `--ID_points_num 200`.

We will **not silently choose between them yet**. That parameter affects the second k-NN filtering stage and will be handled explicitly when we implement NPOS synthesis.


In [2]:
@dataclass
class Config:
    data_root: str = "/kaggle/working/cifar100"

    num_classes: int = 100
    penultimate_dim: int = 512
    feat_dim: int = 128

    epochs: int = 500
    batch_size: int = 256
    learning_rate: float = 0.5
    momentum: float = 0.9
    weight_decay: float = 1e-4

    temperature: float = 0.1
    proto_m: float = 0.95

    w_disp: float = 0.5
    w_comp: float = 1.0
    loss_weight: float = 0.1

    start_epoch_knn: int = 200
    queue_size: int = 600
    sample_from: int = 600
    K: int = 300
    select: int = 200
    cov_mat: float = 0.1
    pick_nums: int = 2

    id_points_num_parser_default: int = 2
    id_points_num_shell_script: int = 200

cfg = Config()
cfg


Config(data_root='/kaggle/working/cifar100', num_classes=100, penultimate_dim=512, feat_dim=128, epochs=500, batch_size=256, learning_rate=0.5, momentum=0.9, weight_decay=0.0001, temperature=0.1, proto_m=0.95, w_disp=0.5, w_comp=1.0, loss_weight=0.1, start_epoch_knn=200, queue_size=600, sample_from=600, K=300, select=200, cov_mat=0.1, pick_nums=2, id_points_num_parser_default=2, id_points_num_shell_script=200)

## 3. Repo-faithful CIFAR-100 transforms

`training_from_scratch/cifar.py` uses a SimCLR-style augmentation and returns **two independently augmented views of each training image**:

```text
RandomResizedCrop(32, scale=(0.2, 1.0))
RandomHorizontalFlip
RandomApply(ColorJitter(...), p=0.8)
RandomGrayscale(p=0.2)
ToTensor
Normalize
```

For CIFAR-100, the repository uses:

$$
\mu=(0.5071, 0.4867, 0.4408)
$$

$$
\sigma=(0.2675, 0.2565, 0.2761)
$$


In [3]:
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)


class TwoCropTransform:
    """Create two independently augmented views of the same image."""

    def __init__(self, transform):
        self.transform = transform

    def __call__(self, x):
        return [self.transform(x), self.transform(x)]


normalize = transforms.Normalize(mean=CIFAR100_MEAN, std=CIFAR100_STD)

train_transform = TwoCropTransform(
    transforms.Compose([
        transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply(
            [transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)],
            p=0.8,
        ),
        transforms.RandomGrayscale(p=0.2),
        transforms.ToTensor(),
        normalize,
    ])
)

test_transform = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])


### Kaggle dataset preparation

Kaggle mounts `/kaggle/input` as read-only. The attached CIFAR-100 dataset contains `train`, `test`, and `meta` directly, while `torchvision.datasets.CIFAR100` expects them inside a `cifar-100-python/` directory.

This cell copies those three files into `/kaggle/working/cifar100/cifar-100-python` once per Kaggle session.


In [4]:
import os
import shutil

src = "/kaggle/input/datasets/alincijov/cifar-100"
root = "/kaggle/working/cifar100"
dst = os.path.join(root, "cifar-100-python")

os.makedirs(dst, exist_ok=True)

for filename in ["train", "test", "meta"]:
    src_file = os.path.join(src, filename)
    dst_file = os.path.join(dst, filename)

    if not os.path.exists(dst_file):
        shutil.copy2(src_file, dst_file)

cfg.data_root = root

print("CIFAR-100 root:", cfg.data_root)
print("Files:", os.listdir(dst))


CIFAR-100 root: /kaggle/working/cifar100
Files: ['meta', 'test', 'train']


## 4. Load CIFAR-100

The repository uses all CIFAR-100 training samples as labeled data (`label_ratio = 1.0`).

For notebook clarity we use `torchvision.datasets.CIFAR100` directly, because the repository wrapper is mainly needed to return two transformed views; our `TwoCropTransform` already provides that behavior.


In [5]:
train_dataset = datasets.CIFAR100(
    root=cfg.data_root,
    train=True,
    download=False,
    transform=train_transform,
)

test_dataset = datasets.CIFAR100(
    root=cfg.data_root,
    train=False,
    download=False,
    transform=test_transform,
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Classes:", len(train_dataset.classes))


Train samples: 50000
Test samples: 10000
Classes: 100


### Check one training batch

Each training sample produces **two views**.

The repository then does:

```python
input = torch.cat([input[0], input[1]], dim=0)
target = target.repeat(2)
```

Therefore with batch size 256, the encoder receives 512 augmented images per iteration.


In [6]:
views, labels = next(iter(train_loader))

view1, view2 = views

print("view1:", view1.shape)
print("view2:", view2.shape)
print("labels:", labels.shape)

images = torch.cat([view1, view2], dim=0)
targets = labels.repeat(2)

print("\nAfter repo-style concatenation:")
print("images:", images.shape)
print("targets:", targets.shape)


view1: torch.Size([256, 3, 32, 32])
view2: torch.Size([256, 3, 32, 32])
labels: torch.Size([256])

After repo-style concatenation:
images: torch.Size([512, 3, 32, 32])
targets: torch.Size([512])


## 5. CIFAR-style ResNet-34 from the repository

This is **not** `torchvision.models.resnet34()`.

The NPOS repo uses a CIFAR-specific stem:

```text
3×3 conv
stride 1
no initial max-pool
```

The block layout is standard ResNet-34:

```text
[3, 4, 6, 3]
```

and the final pooled backbone representation is 512-D.


In [7]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3,
            stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)

        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3,
            stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes, self.expansion * planes,
                    kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = F.relu(out)
        return out


class CIFARResNet(nn.Module):
    def __init__(self, block, num_blocks, in_channel=3, zero_init_residual=False):
        super().__init__()

        self.in_planes = 64

        self.conv1 = nn.Conv2d(
            in_channel, 64, kernel_size=3,
            stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []

        for stride_i in strides:
            layers.append(block(self.in_planes, planes, stride_i))
            self.in_planes = planes * block.expansion

        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        return out


def resnet34():
    return CIFARResNet(BasicBlock, [3, 4, 6, 3])


## 6. NPOS model wrapper

The repository's `SupCEHeadResNet` adds three branches on top of the 512-D ResNet feature:

```text
                         512-D penultimate
                         /       |       \
                        /        |        \
              projection head   fc      OOD MLP
              512→512→128     512→100   512→128→1
```

The normal `forward()` returns the **L2-normalized 128-D projected representation**.

Important: later NPOS synthesis uses the **512-D penultimate feature**, not the 128-D projected feature.


In [8]:
class NPOSResNet34(nn.Module):
    def __init__(self, num_classes=100, feat_dim=128, penultimate_dim=512):
        super().__init__()

        self.encoder = resnet34()

        self.head = nn.Sequential(
            nn.Linear(penultimate_dim, penultimate_dim),
            nn.ReLU(inplace=True),
            nn.Linear(penultimate_dim, feat_dim),
        )

        self.fc = nn.Linear(penultimate_dim, num_classes)

        self.mlp = nn.Sequential(
            nn.Linear(penultimate_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, 1),
        )

        self.prototypes = nn.Parameter(
            torch.zeros(num_classes, feat_dim),
            requires_grad=True,
        )

    def forward(self, x):
        penultimate = self.encoder(x)
        projected = self.head(penultimate)
        return F.normalize(projected, dim=1)


model = NPOSResNet34(
    num_classes=cfg.num_classes,
    feat_dim=cfg.feat_dim,
    penultimate_dim=cfg.penultimate_dim,
).to(device)

# Use both Kaggle T4 GPUs when available.
print("GPU count:", torch.cuda.device_count())

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with nn.DataParallel")

    model.encoder = nn.DataParallel(model.encoder, device_ids=[0, 1])
    model.head = nn.DataParallel(model.head, device_ids=[0, 1])
    model.fc = nn.DataParallel(model.fc, device_ids=[0, 1])
    model.mlp = nn.DataParallel(model.mlp, device_ids=[0, 1])

print("Encoder wrapper:", type(model.encoder).__name__)
print("Head wrapper:   ", type(model.head).__name__)
print("FC wrapper:     ", type(model.fc).__name__)
print("MLP wrapper:    ", type(model.mlp).__name__)

print(model)


GPU count: 2
Using 2 GPUs with nn.DataParallel
Encoder wrapper: DataParallel
Head wrapper:    DataParallel
FC wrapper:      DataParallel
MLP wrapper:     DataParallel
NPOSResNet34(
  (encoder): DataParallel(
    (module): CIFARResNet(
      (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (shortcut): Sequential()
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), 

## 7. Verify every important tensor shape

Expected shapes for a batch of size $B$:

| Tensor | Expected shape |
|---|---|
| input images | `[B, 3, 32, 32]` |
| penultimate | `[B, 512]` |
| projected | `[B, 128]` |
| normalized feature | `[B, 128]` |
| class logits | `[B, 100]` |
| OOD logits | `[B, 1]` |
| prototypes | `[100, 128]` |

We use only a small slice of the batch for this sanity check.


In [ ]:
model.eval()

small_batch = images[:8].to(device)

with torch.no_grad():
    penultimate = model.encoder(small_batch)
    projected = model.head(penultimate)
    normalized_features = F.normalize(projected, dim=1)
    class_logits = model.fc(penultimate)
    ood_logits = model.mlp(penultimate)

print("input:               ", tuple(small_batch.shape))
print("penultimate:         ", tuple(penultimate.shape))
print("projected:           ", tuple(projected.shape))
print("normalized features: ", tuple(normalized_features.shape))
print("class logits:        ", tuple(class_logits.shape))
print("OOD logits:          ", tuple(ood_logits.shape))
print("prototypes:          ", tuple(model.prototypes.shape))

assert penultimate.shape == (8, 512)
assert projected.shape == (8, 128)
assert normalized_features.shape == (8, 128)
assert class_logits.shape == (8, 100)
assert ood_logits.shape == (8, 1)
assert model.prototypes.shape == (100, 128)

print("\nAll tensor-shape checks passed.")


## Foundation complete

At this point we have the data/model foundation needed by the CIFAR-100 NPOS experiment:

```text
CIFAR-100
    ↓
two augmented views
    ↓
CIFAR ResNet-34
    ↓
512-D penultimate feature
    ├── classifier: 512 → 100
    ├── projection head: 512 → 512 → 128
    └── OOD MLP: 512 → 128 → 1
```

### Next stage

Next we should implement:

1. prototype initialization
2. `CompLoss`
3. `DispLoss`
4. one-batch closed-set training verification

Only after those behave correctly should we add the 512-D class queues and the two-stage FAISS NPOS synthesis.


## 8. `CompLoss`

`CompLoss` operates on the **L2-normalized 128-D projected features**. Each sample is compared against all 100 class prototypes with temperature-scaled cosine similarity, and the correct prototype is the positive target.


In [ ]:
class CompLoss(nn.Module):
    def __init__(self, num_classes, temperature=0.1, base_temperature=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.temperature = temperature
        self.base_temperature = base_temperature

    def forward(self, features, prototypes, labels):
        device = features.device

        proxy_labels = torch.arange(
            self.num_classes,
            device=device
        )

        batch_size = features.shape[0]
        labels = labels.contiguous().view(-1, 1)

        if labels.shape[0] != batch_size:
            raise ValueError(
                "Number of labels does not match number of features"
            )

        mask = torch.eq(
            labels,
            proxy_labels.view(1, -1)
        ).float()

        contrast_feature = F.normalize(
            prototypes,
            dim=1
        )

        logits = (
            features @ contrast_feature.T
        ) / self.temperature

        logits_max, _ = torch.max(
            logits,
            dim=1,
            keepdim=True
        )
        logits = logits - logits_max.detach()

        exp_logits = torch.exp(logits)
        log_prob = logits - torch.log(
            exp_logits.sum(dim=1, keepdim=True)
        )

        mean_log_prob_pos = (
            mask * log_prob
        ).sum(dim=1)

        loss = -(
            self.temperature / self.base_temperature
        ) * mean_log_prob_pos.mean()

        return loss


## 9. Initialize class prototypes

For strict repo fidelity, the initial 128-D class prototypes are computed by running the model on the test/validation loader, averaging features within each class, and L2-normalizing each class mean.


In [ ]:
@torch.no_grad()
def initialize_prototypes(
    model,
    loader,
    num_classes=100,
    feat_dim=128,
):
    model.eval()

    prototypes = torch.zeros(
        num_classes,
        feat_dim,
        device=device
    )

    counts = torch.zeros(
        num_classes,
        device=device
    )

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        features = model(batch_images)

        for cls in batch_labels.unique():
            cls_idx = int(cls.item())
            mask = batch_labels == cls_idx

            prototypes[cls_idx] += features[mask].sum(dim=0)
            counts[cls_idx] += mask.sum()

    if torch.any(counts == 0):
        missing = torch.where(counts == 0)[0].tolist()
        raise RuntimeError(
            f"Classes with no samples during prototype init: {missing}"
        )

    prototypes = prototypes / counts.unsqueeze(1)
    prototypes = F.normalize(prototypes, dim=1)

    model.train()
    return prototypes


prototypes = initialize_prototypes(
    model,
    test_loader,
    num_classes=cfg.num_classes,
    feat_dim=cfg.feat_dim,
)

print("Prototype shape:", tuple(prototypes.shape))
print(
    "Mean prototype norm:",
    prototypes.norm(dim=1).mean().item()
)


## 10. `DispLoss`

`DispLoss` momentum-updates each observed class prototype and penalizes similarity between different prototypes.


In [ ]:
class DispLoss(nn.Module):
    def __init__(
        self,
        num_classes,
        feat_dim,
        proto_m=0.95,
        temperature=0.1,
        base_temperature=0.1,
    ):
        super().__init__()

        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.proto_m = proto_m
        self.temperature = temperature
        self.base_temperature = base_temperature

    def forward(self, features, labels, prototypes):
        updated = prototypes.detach().clone()

        with torch.no_grad():
            for j in range(len(features)):
                cls = int(labels[j].item())

                updated[cls] = F.normalize(
                    updated[cls] * self.proto_m
                    + features[j].detach() * (1 - self.proto_m),
                    dim=0
                )

        logits = (
            updated @ updated.T
        ) / self.temperature

        mask = 1.0 - torch.eye(
            self.num_classes,
            device=features.device
        )

        mean_prob_neg = torch.log(
            (mask * torch.exp(logits)).sum(dim=1)
            / mask.sum(dim=1)
        )

        mean_prob_neg = mean_prob_neg[
            ~torch.isnan(mean_prob_neg)
        ]

        loss = (
            self.temperature / self.base_temperature
        ) * mean_prob_neg.mean()

        return loss, updated


## 11. One-batch loss sanity check

This verifies that `CompLoss` and `DispLoss` are finite before we build the full training loop.


In [ ]:
comp_criterion = CompLoss(
    num_classes=cfg.num_classes,
    temperature=cfg.temperature,
    base_temperature=cfg.temperature,
)

disp_criterion = DispLoss(
    num_classes=cfg.num_classes,
    feat_dim=cfg.feat_dim,
    proto_m=cfg.proto_m,
    temperature=cfg.temperature,
    base_temperature=cfg.temperature,
)

views, labels = next(iter(train_loader))
view1, view2 = views

batch_images = torch.cat(
    [view1, view2],
    dim=0
).to(device)

batch_targets = labels.repeat(2).to(device)

model.train()

penultimate = model.encoder(batch_images)
projected = model.head(penultimate)
features = F.normalize(projected, dim=1)

comp_loss = comp_criterion(
    features,
    prototypes,
    batch_targets
)

disp_loss, updated_prototypes = disp_criterion(
    features,
    batch_targets,
    prototypes
)

print("Images:      ", tuple(batch_images.shape))
print("Penultimate: ", tuple(penultimate.shape))
print("Features:    ", tuple(features.shape))
print("Targets:     ", tuple(batch_targets.shape))
print("CompLoss:    ", float(comp_loss.detach().cpu()))
print("DispLoss:    ", float(disp_loss.detach().cpu()))
print(
    "Prototype mean norm:",
    float(updated_prototypes.norm(dim=1).mean().cpu())
)

assert torch.isfinite(comp_loss)
assert torch.isfinite(disp_loss)

print("\nLoss sanity checks passed.")


## 12. One backward-pass test

The closed-set objective is:

$$
L_{closed} = w_{disp}L_{disp} + w_{comp}L_{comp}
$$

This cell verifies that the encoder and projection head receive gradients.


In [ ]:
optimizer = torch.optim.SGD(
    [
        {"params": model.encoder.parameters()},
        {"params": model.fc.parameters()},
        {"params": model.head.parameters()},
        {
            "params": model.mlp.parameters(),
            "lr": cfg.learning_rate * 0.1,
        },
    ],
    lr=cfg.learning_rate,
    momentum=cfg.momentum,
    nesterov=True,
    weight_decay=cfg.weight_decay,
)

closed_loss = (
    cfg.w_disp * disp_loss
    + cfg.w_comp * comp_loss
)

optimizer.zero_grad()
closed_loss.backward()


def grad_summary(module):
    grads = [
        p.grad for p in module.parameters()
        if p.requires_grad and p.grad is not None
    ]

    if not grads:
        return {
            "params_with_grad": 0,
            "mean_abs_grad": None
        }

    return {
        "params_with_grad": len(grads),
        "mean_abs_grad": torch.stack([
            g.detach().abs().mean().cpu()
            for g in grads
        ]).mean().item()
    }


print("Closed loss:", float(closed_loss.detach().cpu()))
print("Encoder gradients:", grad_summary(model.encoder))
print("Head gradients:   ", grad_summary(model.head))
print("FC gradients:     ", grad_summary(model.fc))
print("OOD MLP gradients:", grad_summary(model.mlp))

optimizer.step()

prototypes = updated_prototypes.detach()

print("\nBackward + optimizer step completed.")


## Closed-set representation stage complete

If all cells above run successfully, the next stage is the class-wise NPOS feature queue:

```text
[100 classes, 600 queue entries, 512-D penultimate features]
```

We will verify the queue before adding FAISS.


## 13. Class-wise 512-D feature queue

NPOS does **not** store the 128-D projected representation in its synthesis queue.

The repository stores the raw ResNet penultimate feature:

```text
ResNet-34
   ↓
512-D penultimate feature
   ↓
class-specific queue
```

For CIFAR-100, the paper configuration uses 600 queue entries per class, so the full memory has shape:

```text
[100, 600, 512]
```

Each class owns its own FIFO queue. Before a class reaches 600 entries, new features fill unused slots. After the queue is full, the oldest feature is removed and the newest feature is appended.


In [ ]:
class ClassFeatureQueue:
    def __init__(
        self,
        num_classes=100,
        queue_size=600,
        feature_dim=512,
        device=device,
    ):
        self.num_classes = num_classes
        self.queue_size = queue_size
        self.feature_dim = feature_dim
        self.device = device

        self.data = torch.zeros(
            num_classes,
            queue_size,
            feature_dim,
            device=device,
        )

        # Number of valid entries currently stored per class.
        self.counts = torch.zeros(
            num_classes,
            dtype=torch.long,
            device=device,
        )

    @torch.no_grad()
    def update(self, features, labels):
        """
        features: [B, 512] penultimate features
        labels:   [B]
        """

        if features.ndim != 2:
            raise ValueError(
                f"Expected [B, D] features, got {tuple(features.shape)}"
            )

        if features.shape[1] != self.feature_dim:
            raise ValueError(
                f"Expected feature_dim={self.feature_dim}, "
                f"got {features.shape[1]}"
            )

        if labels.shape[0] != features.shape[0]:
            raise ValueError(
                "Number of labels must match number of features"
            )

        for feature, label in zip(features.detach(), labels):
            cls = int(label.item())
            count = int(self.counts[cls].item())

            if count < self.queue_size:
                self.data[cls, count] = feature
                self.counts[cls] += 1
            else:
                # FIFO update, matching the repository's logic:
                # drop oldest entry and append newest entry.
                self.data[cls, :-1] = self.data[cls, 1:].clone()
                self.data[cls, -1] = feature

    def class_features(self, cls):
        cls = int(cls)
        count = int(self.counts[cls].item())
        return self.data[cls, :count]

    def is_class_full(self, cls):
        return int(self.counts[int(cls)].item()) >= self.queue_size

    def all_classes_full(self):
        return bool(torch.all(self.counts >= self.queue_size).item())

    def summary(self):
        counts_cpu = self.counts.detach().cpu()
        return {
            "shape": tuple(self.data.shape),
            "min_count": int(counts_cpu.min()),
            "max_count": int(counts_cpu.max()),
            "mean_count": float(counts_cpu.float().mean()),
            "full_classes": int((counts_cpu >= self.queue_size).sum()),
        }


feature_queue = ClassFeatureQueue(
    num_classes=cfg.num_classes,
    queue_size=cfg.queue_size,
    feature_dim=cfg.penultimate_dim,
    device=device,
)

print(feature_queue.summary())


## 14. Fill the queue with penultimate features

We will now run several training batches through the encoder and update the queue using:

```python
penultimate = model.encoder(images)
feature_queue.update(penultimate, targets)
```

The important check is that the queue receives `[B, 512]` features, not `[B, 128]` projected embeddings.


In [ ]:
@torch.no_grad()
def fill_queue_for_batches(
    model,
    loader,
    feature_queue,
    num_batches=20,
):
    model.eval()

    for batch_idx, (views, labels) in enumerate(loader):
        if batch_idx >= num_batches:
            break

        view1, view2 = views

        batch_images = torch.cat(
            [view1, view2],
            dim=0
        ).to(device)

        batch_targets = labels.repeat(2).to(device)

        penultimate = model.encoder(batch_images)

        # This assertion protects against accidentally using 128-D features.
        assert penultimate.shape[1] == cfg.penultimate_dim

        feature_queue.update(
            penultimate,
            batch_targets
        )

    model.train()


fill_queue_for_batches(
    model,
    train_loader,
    feature_queue,
    num_batches=20,
)

print("Queue summary after 20 batches:")
print(feature_queue.summary())


## 15. Inspect queue statistics

We now inspect:

- the queue tensor shape
- the number of stored features per class
- min / max / mean fill count
- the number of classes already full
- feature-norm statistics

Because CIFAR-100 is balanced, the counts should become reasonably similar across classes after enough shuffled batches.


In [ ]:
counts = feature_queue.counts.detach().cpu()

print("Queue tensor shape:", tuple(feature_queue.data.shape))
print("Expected shape:    ", (100, 600, 512))
print()

print("Minimum class count:", int(counts.min()))
print("Maximum class count:", int(counts.max()))
print("Mean class count:   ", float(counts.float().mean()))
print(
    "Full classes:       ",
    int((counts >= cfg.queue_size).sum())
)

print("\nFirst 20 class counts:")
print(counts[:20].tolist())

assert feature_queue.data.shape == (
    cfg.num_classes,
    cfg.queue_size,
    cfg.penultimate_dim,
)

assert feature_queue.data.shape[-1] == 512

print("\nQueue shape checks passed.")


In [ ]:
valid_features = []

for cls in range(cfg.num_classes):
    cls_features = feature_queue.class_features(cls)

    if len(cls_features) > 0:
        valid_features.append(cls_features)

valid_features = torch.cat(valid_features, dim=0)

feature_norms = valid_features.norm(dim=1)

print("Stored valid features:", valid_features.shape)
print(
    "Feature norm mean:",
    float(feature_norms.mean().cpu())
)
print(
    "Feature norm std:",
    float(feature_norms.std().cpu())
)
print(
    "Feature norm min/max:",
    float(feature_norms.min().cpu()),
    float(feature_norms.max().cpu())
)

print(
    "\nThese are raw 512-D penultimate features, "
    "so their norms are NOT expected to equal 1."
)


## 16. Verify FIFO behavior on one class

Before moving to FAISS, we should verify the queue really behaves like FIFO once full.

This test uses a tiny synthetic queue with one class, queue size 3, and feature dimension 2. It does **not** modify the real CIFAR-100 feature queue.


In [ ]:
toy_queue = ClassFeatureQueue(
    num_classes=1,
    queue_size=3,
    feature_dim=2,
    device=device,
)

toy_features_1 = torch.tensor(
    [
        [1.0, 1.0],
        [2.0, 2.0],
        [3.0, 3.0],
    ],
    device=device,
)

toy_labels = torch.zeros(
    3,
    dtype=torch.long,
    device=device,
)

toy_queue.update(
    toy_features_1,
    toy_labels
)

print("After filling:")
print(toy_queue.class_features(0).cpu())

toy_queue.update(
    torch.tensor([[4.0, 4.0]], device=device),
    torch.tensor([0], dtype=torch.long, device=device),
)

print("\nAfter adding [4, 4]:")
print(toy_queue.class_features(0).cpu())

expected = torch.tensor(
    [
        [2.0, 2.0],
        [3.0, 3.0],
        [4.0, 4.0],
    ]
)

assert torch.allclose(
    toy_queue.class_features(0).cpu(),
    expected
)

print("\nFIFO check passed.")


## Queue stage complete

If all cells above run successfully, we have verified the exact memory representation NPOS needs:

```text
100 class-specific queues
×
600 entries per class
×
512-D penultimate features
```

### Next stage

Next we will implement **only the first k-NN stage**:

```text
one class queue
    ↓
L2-normalize its 512-D features
    ↓
build FAISS index
    ↓
search K = 300 neighbors
    ↓
take the K-th-neighbor distance
    ↓
select top 200 largest distances
    ↓
boundary candidate set
```

We will inspect those distances and boundary points before adding Gaussian outlier synthesis.


## 17. First FAISS k-NN boundary-selection stage

We now reproduce the first k-NN stage from `training_from_scratch/KNN.py`.

For one CIFAR-100 class:

```text
valid 512-D queue features
    ↓
L2-normalize
    ↓
FAISS index
    ↓
search K = 300 neighbors
    ↓
take the K-th-neighbor distance for each ID feature
    ↓
rank distances
    ↓
select top 200 largest distances
    ↓
boundary candidate set
```

A larger K-th-neighbor distance means the point lies in a lower-density region of the class feature distribution.


### FAISS setup

The repository imports:

```python
import faiss
import faiss.contrib.torch_utils
```

and searches normalized Torch tensors directly.

On Kaggle, `faiss-cpu` is often already available. This section first checks the installed FAISS package and uses `IndexFlatL2`, which matches the repository's L2-distance search behavior.


In [ ]:
import faiss

print("FAISS version:", getattr(faiss, "__version__", "unknown"))

try:
    import faiss.contrib.torch_utils
    print("faiss.contrib.torch_utils available")
except Exception as e:
    print("faiss.contrib.torch_utils not available:", e)


## 18. Make sure one class has enough queue entries

The paper/repo configuration uses:

```text
queue size = 600
K = 300
select = 200
```

For a faithful first test, we want one class with at least 600 valid entries.

If the earlier 20-batch diagnostic did not fill a class completely, this cell continues feeding batches into the queue until at least one class reaches 600 entries.


In [ ]:
@torch.no_grad()
def fill_until_one_class_full(
    model,
    loader,
    feature_queue,
):
    if feature_queue.all_classes_full():
        return

    model.eval()

    for views, labels in loader:
        full_classes = torch.where(
            feature_queue.counts >= feature_queue.queue_size
        )[0]

        if len(full_classes) > 0:
            break

        view1, view2 = views

        batch_images = torch.cat(
            [view1, view2],
            dim=0
        ).to(device)

        batch_targets = labels.repeat(2).to(device)

        penultimate = model.encoder(batch_images)

        assert penultimate.shape[1] == cfg.penultimate_dim

        feature_queue.update(
            penultimate,
            batch_targets
        )

    model.train()


fill_until_one_class_full(
    model,
    train_loader,
    feature_queue,
)

counts = feature_queue.counts.detach().cpu()

full_classes = torch.where(
    counts >= cfg.queue_size
)[0]

print("Number of full classes:", len(full_classes))
print("First full classes:", full_classes[:10].tolist())

if len(full_classes) == 0:
    print(
        "No class is full yet. Run this cell again or use the "
        "all-classes filling cell below."
    )


### Optional: fill all 100 queues

The actual training script waits until the class queues are populated before NPOS synthesis is used.

For a more complete diagnostic, this helper can continue through the training loader until every class reaches 600 entries.

This may take a little longer than filling one class, but it remains much cheaper than full training.


In [ ]:
@torch.no_grad()
def fill_all_class_queues(
    model,
    loader,
    feature_queue,
    max_passes=5,
):
    model.eval()

    for pass_idx in range(max_passes):
        if feature_queue.all_classes_full():
            break

        for views, labels in loader:
            if feature_queue.all_classes_full():
                break

            view1, view2 = views

            batch_images = torch.cat(
                [view1, view2],
                dim=0
            ).to(device)

            batch_targets = labels.repeat(2).to(device)

            penultimate = model.encoder(batch_images)

            feature_queue.update(
                penultimate,
                batch_targets
            )

        print(
            f"Pass {pass_idx + 1}:",
            feature_queue.summary()
        )

    model.train()

    return feature_queue.all_classes_full()


# Leave this commented if you only want the single-class test first.
# all_full = fill_all_class_queues(model, train_loader, feature_queue)
# print("All queues full:", all_full)


## 19. Repo-style `KNN_dis_search_decrease`

The repository's function does three important things:

1. L2-normalize the query features
2. use FAISS to search for `K` neighbors
3. select the features with the **largest** K-th-neighbor distances using `torch.topk`

Despite the variable name `minD_idx` in the repo, `torch.topk` uses `largest=True` by default.


In [ ]:
def knn_dis_search_decrease(
    target,
    index,
    K=300,
    select=200,
):
    # Repo-style L2 normalization.
    target_norm = torch.norm(
        target,
        p=2,
        dim=1,
        keepdim=True
    )

    normed_target = target / target_norm

    # For notebook portability, send normalized features to CPU NumPy.
    # IndexFlatL2 then returns squared L2 distances.
    query_np = (
        normed_target
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    distances, output_indices = index.search(
        query_np,
        K
    )

    distances = torch.from_numpy(
        distances
    )

    output_indices = torch.from_numpy(
        output_indices
    )

    kth_distance = distances[:, -1]

    selected_distances, selected_idx = torch.topk(
        kth_distance,
        select
    )

    return {
        "selected_idx": selected_idx,
        "selected_distances": selected_distances,
        "kth_distance": kth_distance,
        "all_distances": distances,
        "neighbor_indices": output_indices,
        "normed_target": normed_target.detach().cpu(),
    }


## 20. Build the FAISS index for one full class

We select the first class whose queue has 600 valid entries.

The FAISS index contains the **normalized 512-D queue features** for that class only.


In [ ]:
counts = feature_queue.counts.detach().cpu()

full_classes = torch.where(
    counts >= cfg.queue_size
)[0]

if len(full_classes) == 0:
    raise RuntimeError(
        "No class has 600 queue entries yet. "
        "Run the previous filling cell again."
    )

selected_class = int(full_classes[0].item())

class_features = feature_queue.class_features(
    selected_class
).detach()

print("Selected class:", selected_class)
print("Raw class feature shape:", tuple(class_features.shape))

assert class_features.shape == (
    cfg.queue_size,
    cfg.penultimate_dim
)

class_features_norm = F.normalize(
    class_features,
    dim=1
)

index = faiss.IndexFlatL2(
    cfg.penultimate_dim
)

index.add(
    class_features_norm
    .float()
    .cpu()
    .numpy()
)

print("FAISS dimension:", index.d)
print("FAISS index size:", index.ntotal)

assert index.ntotal == cfg.queue_size


## 21. Search `K = 300` neighbors and select the top 200 boundary candidates

For every one of the 600 class features, we compute its 300th-nearest-neighbor distance.

Then we select the 200 samples with the largest such distances.


In [ ]:
boundary_result = knn_dis_search_decrease(
    target=class_features,
    index=index,
    K=cfg.K,
    select=cfg.select,
)

kth_distance = boundary_result["kth_distance"]
boundary_idx = boundary_result["selected_idx"]
boundary_distances = boundary_result["selected_distances"]

print("K:", cfg.K)
print("Select:", cfg.select)

print("\nk-th distance shape:", tuple(kth_distance.shape))
print("Boundary index shape:", tuple(boundary_idx.shape))
print("Boundary distance shape:", tuple(boundary_distances.shape))

print("\nAll k-th distance statistics:")
print("  min: ", float(kth_distance.min()))
print("  mean:", float(kth_distance.mean()))
print("  max: ", float(kth_distance.max()))

print("\nSelected boundary-distance statistics:")
print("  min: ", float(boundary_distances.min()))
print("  mean:", float(boundary_distances.mean()))
print("  max: ", float(boundary_distances.max()))

print("\nFirst 20 boundary indices:")
print(boundary_idx[:20].tolist())

assert kth_distance.shape == (cfg.queue_size,)
assert boundary_idx.shape == (cfg.select,)
assert boundary_distances.shape == (cfg.select,)

print("\nBoundary-selection shape checks passed.")


## 22. Verify that boundary candidates really have larger k-NN distances

If the selection is working correctly:

```text
mean boundary K-th distance
>
mean K-th distance across all ID points
```

We also compute the cutoff distance corresponding to the 200th selected boundary point.


In [ ]:
all_mean = kth_distance.mean()
boundary_mean = boundary_distances.mean()
cutoff = boundary_distances.min()

print(
    "Mean K-th distance, all ID points:",
    float(all_mean)
)

print(
    "Mean K-th distance, boundary set:",
    float(boundary_mean)
)

print(
    "Boundary cutoff distance:",
    float(cutoff)
)

assert boundary_mean >= all_mean

num_at_or_above_cutoff = int(
    (kth_distance >= cutoff).sum()
)

print(
    "Number of points at/above cutoff:",
    num_at_or_above_cutoff
)

print("\nBoundary-density check passed.")


## 23. Visualize the K-th-neighbor distance distribution

The histogram below shows the distribution of the 300th-neighbor distance for all 600 queue features.

The vertical dashed line marks the cutoff for the top 200 boundary candidates.

Because FAISS `IndexFlatL2` returns **squared Euclidean distances**, the horizontal axis is squared L2 distance between normalized features.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.hist(
    kth_distance.numpy(),
    bins=40,
    alpha=0.8
)

plt.axvline(
    float(cutoff),
    linestyle="--",
    linewidth=2,
    label=f"Top-{cfg.select} cutoff"
)

plt.xlabel(
    f"{cfg.K}-th neighbor squared L2 distance"
)
plt.ylabel("Number of ID features")
plt.title(
    f"Class {selected_class}: k-NN boundary score distribution"
)
plt.legend()
plt.show()


## 24. Inspect the selected boundary features

The selected boundary features are still **real ID features** from the class queue.

They are not synthetic OOD samples yet.


In [ ]:
boundary_features = class_features[
    boundary_idx.to(class_features.device)
]

print(
    "Boundary feature shape:",
    tuple(boundary_features.shape)
)

print(
    "Raw boundary feature norm mean:",
    float(
        boundary_features
        .norm(dim=1)
        .mean()
        .cpu()
    )
)

assert boundary_features.shape == (
    cfg.select,
    cfg.penultimate_dim
)

print(
    "\nThese 200 features are the low-density "
    "ID boundary candidate pool."
)


## First k-NN stage complete

We have now reproduced the first NPOS geometric operation:

```text
600 class-specific 512-D ID features
    ↓
L2 normalization
    ↓
FAISS K = 300 search
    ↓
300th-neighbor distance
    ↓
top 200 largest distances
    ↓
200 low-density boundary candidates
```

### Next stage

The repo does **not** synthesize around all 200 boundary candidates.

It randomly chooses `pick_nums` boundary anchors from this pool, creates Gaussian perturbations around them, and then runs a **second k-NN filtering stage**.

The next notebook version will implement:

```text
top-200 boundary pool
    ↓
randomly choose boundary anchor(s)
    ↓
Gaussian perturbations in 512-D
    ↓
candidate synthetic features
```

We will inspect the generated candidates before implementing the second rejection/filtering step.


## 25. Important paper-vs-repo configuration discrepancy

Before Gaussian synthesis, there are two configuration mismatches worth making explicit.

### Queue size

The paper reports a **class queue size of 600**, which is what this notebook currently uses:

```text
queue_size = 600
```

However, the raw `train_CIFAR100.py` parser defaults to:

```text
sample_number = 1000
```

and the provided `train_npos_cifar100.sh` does **not** override `sample_number`.

So:

```text
paper setting:      queue = 600
raw shell execution: queue = 1000
```

### Prototype momentum

The paper reports:

```text
proto_m = 0.95
```

but `train_CIFAR100.py` defaults to:

```text
proto_m = 0.5
```

and the shell script again does not override it.

For this notebook we continue with the **paper configuration** (`queue_size=600`, `proto_m=0.95`) because the goal is to reproduce the reported CIFAR-100 experiment rather than blindly reproduce parser defaults that conflict with the paper.

We will keep noting such differences instead of silently changing them.


## 26. Gaussian proposal generation around boundary anchors

The repo's `generate_outliers()` first obtains the top `select=200` low-density ID boundary candidates.

It then randomly chooses only:

```text
pick_nums = 2
```

anchors from that boundary pool.

For each selected anchor, it adds `sample_from=600` Gaussian perturbations in the **512-D penultimate feature space**.

The proposal rule implemented by the repo is:

$$
x_{candidate} = x_{boundary} + 0.1\epsilon
$$

where:

$$
\epsilon \sim \mathcal{N}(0, I_{512})
$$

At this stage we generate the candidate cloud only. We do **not** yet apply the second k-NN filtering step.


In [ ]:
def sample_boundary_anchors(
    boundary_features,
    pick_nums=2,
    seed=None,
):
    if pick_nums > len(boundary_features):
        raise ValueError(
            "pick_nums cannot exceed number of boundary candidates"
        )

    if seed is not None:
        rng = np.random.default_rng(seed)
        anchor_indices = rng.choice(
            len(boundary_features),
            size=pick_nums,
            replace=False,
        )
    else:
        anchor_indices = np.random.choice(
            len(boundary_features),
            size=pick_nums,
            replace=False,
        )

    anchor_indices = torch.as_tensor(
        anchor_indices,
        dtype=torch.long,
        device=boundary_features.device,
    )

    anchors = boundary_features[anchor_indices]

    return anchor_indices, anchors


anchor_indices, boundary_anchors = sample_boundary_anchors(
    boundary_features,
    pick_nums=cfg.pick_nums,
    seed=SEED,
)

print("Boundary pool shape:", tuple(boundary_features.shape))
print("Selected anchor indices within boundary pool:", anchor_indices.cpu().tolist())
print("Selected anchor shape:", tuple(boundary_anchors.shape))

assert boundary_anchors.shape == (
    cfg.pick_nums,
    cfg.penultimate_dim,
)


## 27. Generate the standard Gaussian noise

The training script creates one set of `sample_from=600` standard Gaussian vectors:

```python
MultivariateNormal(
    zeros(512),
    eye(512)
).rsample((600,))
```

Then `generate_outliers()` repeats that noise set for each selected boundary anchor.


In [ ]:
torch.manual_seed(SEED)

negative_samples = torch.randn(
    cfg.sample_from,
    cfg.penultimate_dim,
    device=device,
)

print("Gaussian noise shape:", tuple(negative_samples.shape))
print(
    "Noise mean:",
    float(negative_samples.mean().cpu())
)
print(
    "Noise std:",
    float(negative_samples.std().cpu())
)

assert negative_samples.shape == (
    cfg.sample_from,
    cfg.penultimate_dim,
)


## 28. Create the 512-D candidate synthetic features

The repo repeats each selected boundary anchor `sample_from` times and adds scaled Gaussian noise.

With:

```text
pick_nums   = 2
sample_from = 600
```

we expect:

```text
2 × 600 = 1200 candidate features
```

for this class before the second k-NN filtering stage.


In [ ]:
def generate_gaussian_candidates(
    anchors,
    negative_samples,
    cov_mat=0.1,
):
    num_anchors = anchors.shape[0]
    num_noise = negative_samples.shape[0]

    # [A, 1, D] + [1, N, D] -> [A, N, D]
    candidates = (
        anchors[:, None, :]
        +
        cov_mat * negative_samples[None, :, :]
    )

    # Flatten to repo-style [A*N, D].
    candidates = candidates.reshape(
        num_anchors * num_noise,
        -1
    )

    return candidates


candidate_features = generate_gaussian_candidates(
    boundary_anchors,
    negative_samples,
    cov_mat=cfg.cov_mat,
)

print("Selected anchors:", cfg.pick_nums)
print("Noise samples per anchor:", cfg.sample_from)
print("Candidate feature shape:", tuple(candidate_features.shape))

expected_candidates = (
    cfg.pick_nums * cfg.sample_from
)

assert candidate_features.shape == (
    expected_candidates,
    cfg.penultimate_dim,
)

print(
    f"\nGenerated {expected_candidates} candidate "
    "synthetic features for one class."
)


## 29. Verify candidate displacement from each anchor

Because the proposal is:

```text
candidate = anchor + 0.1 × noise
```

the candidate cloud should remain local to each boundary anchor.

We measure the Euclidean displacement from each proposal to the anchor that generated it.


In [ ]:
candidate_by_anchor = candidate_features.view(
    cfg.pick_nums,
    cfg.sample_from,
    cfg.penultimate_dim,
)

displacements = (
    candidate_by_anchor
    - boundary_anchors[:, None, :]
).norm(dim=2)

print("Displacement tensor shape:", tuple(displacements.shape))
print(
    "Mean displacement:",
    float(displacements.mean().cpu())
)
print(
    "Std displacement:",
    float(displacements.std().cpu())
)
print(
    "Min / max displacement:",
    float(displacements.min().cpu()),
    float(displacements.max().cpu())
)

print(
    "\nExpected scale is roughly 0.1 * sqrt(512) ≈",
    0.1 * (cfg.penultimate_dim ** 0.5)
)


## 30. Compare raw feature norms

The queue and synthetic candidates live in the **raw 512-D penultimate space**.

The candidate generation itself does not normalize them. Normalization happens only when computing k-NN distances.


In [ ]:
anchor_norms = boundary_anchors.norm(dim=1)
candidate_norms = candidate_features.norm(dim=1)

print(
    "Anchor norm mean:",
    float(anchor_norms.mean().cpu())
)
print(
    "Candidate norm mean:",
    float(candidate_norms.mean().cpu())
)
print(
    "Candidate norm std:",
    float(candidate_norms.std().cpu())
)

print(
    "\nCandidates remain raw 512-D features; "
    "they are not unit-normalized here."
)


## 31. PCA visualization of one class

To make the proposal mechanism easier to inspect, we project:

- all 600 ID queue features
- the 200 boundary candidates
- the 2 selected boundary anchors
- a subset of Gaussian candidate features

from 512-D to 2-D using PCA.

This visualization is diagnostic only; PCA is **not** part of NPOS training.


In [ ]:
from sklearn.decomposition import PCA

id_np = class_features.detach().cpu().numpy()
boundary_np = boundary_features.detach().cpu().numpy()
anchors_np = boundary_anchors.detach().cpu().numpy()

# Keep the plot readable.
num_plot_candidates = min(
    400,
    len(candidate_features)
)

candidate_subset_np = (
    candidate_features[:num_plot_candidates]
    .detach()
    .cpu()
    .numpy()
)

combined = np.concatenate(
    [
        id_np,
        boundary_np,
        anchors_np,
        candidate_subset_np,
    ],
    axis=0,
)

pca = PCA(n_components=2)
combined_2d = pca.fit_transform(combined)

n_id = len(id_np)
n_boundary = len(boundary_np)
n_anchor = len(anchors_np)

id_2d = combined_2d[:n_id]
boundary_2d = combined_2d[
    n_id:n_id + n_boundary
]
anchors_2d = combined_2d[
    n_id + n_boundary:
    n_id + n_boundary + n_anchor
]
candidates_2d = combined_2d[
    n_id + n_boundary + n_anchor:
]

print(
    "PCA explained variance ratio:",
    pca.explained_variance_ratio_
)


In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    id_2d[:, 0],
    id_2d[:, 1],
    s=12,
    alpha=0.35,
    label="ID queue",
)

plt.scatter(
    boundary_2d[:, 0],
    boundary_2d[:, 1],
    s=20,
    alpha=0.6,
    label="Top-200 boundary",
)

plt.scatter(
    candidates_2d[:, 0],
    candidates_2d[:, 1],
    s=14,
    alpha=0.4,
    label="Gaussian candidates",
)

plt.scatter(
    anchors_2d[:, 0],
    anchors_2d[:, 1],
    s=140,
    marker="X",
    label="Selected anchors",
)

plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.title(
    f"Class {selected_class}: ID boundary and Gaussian proposals"
)
plt.legend()
plt.show()


## Gaussian proposal stage complete

We now have:

```text
600 ID queue features
    ↓
top-200 low-density boundary pool
    ↓
randomly choose 2 boundary anchors
    ↓
600 Gaussian perturbations per anchor
    ↓
1200 candidate synthetic features
```

These 1200 candidates are **not yet the final NPOS outliers**.

### Next stage: second k-NN filtering

The repository next:

```text
1200 candidates
    ↓
L2-normalize
    ↓
search against the same class ID FAISS index
    ↓
compute K-th-neighbor distance
    ↓
keep candidates with the largest distances
```

That second stage is the actual rejection/filtering step that turns Gaussian proposals into retained virtual OOD features.

We will also resolve the `ID_points_num` discrepancy explicitly there:

```text
train_CIFAR100.py default: 2
train_npos_cifar100.sh:    200
```


## 32. Second k-NN filtering stage

Now we reproduce the second k-NN operation from `training_from_scratch/KNN.py`.

The 1200 Gaussian proposals are **not** all retained. NPOS measures how far each proposal lies from the ID class distribution:

```text
1200 candidate 512-D features
    ↓
L2-normalize candidates
    ↓
search against the same normalized class FAISS index
    ↓
take each candidate's K-th-neighbor distance
    ↓
retain candidates with the largest distances
```

Large K-th-neighbor distance again means lower estimated ID density.

### Important `ID_points_num` discrepancy

The raw parser in `train_CIFAR100.py` says:

```text
ID_points_num = 2
```

but the provided `train_npos_cifar100.sh` explicitly launches:

```text
--ID_points_num 200
```

Therefore the **actual provided CIFAR-100 launch command uses 200**, not 2.

With `pick_nums=2`, the shell-script path attempts to retain:

```text
200 candidates × 2 selected anchors = 400 synthetic OOD features per class
```

We will use `200` as the repo launch setting below, while keeping the parser-default result available for comparison.


## 33. Compute K-th-neighbor distances for all Gaussian proposals

This part is straightforward: normalize every candidate and query the same class-specific FAISS index built from normalized ID features.


In [ ]:
def candidate_knn_distances(
    candidates,
    index,
    K=300,
):
    normed_candidates = F.normalize(
        candidates,
        dim=1
    )

    candidate_np = (
        normed_candidates
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    distances, neighbor_indices = index.search(
        candidate_np,
        K
    )

    distances = torch.from_numpy(distances)
    neighbor_indices = torch.from_numpy(neighbor_indices)

    kth_distance = distances[:, -1]

    return {
        "normed_candidates": normed_candidates.detach().cpu(),
        "all_distances": distances,
        "neighbor_indices": neighbor_indices,
        "kth_distance": kth_distance,
    }


candidate_knn = candidate_knn_distances(
    candidate_features,
    index,
    K=cfg.K,
)

candidate_kth = candidate_knn["kth_distance"]

print("Candidate K-th distance shape:", tuple(candidate_kth.shape))
print("Expected:", (cfg.pick_nums * cfg.sample_from,))

print("\nCandidate K-th distance statistics:")
print("  min: ", float(candidate_kth.min()))
print("  mean:", float(candidate_kth.mean()))
print("  max: ", float(candidate_kth.max()))

assert candidate_kth.shape == (
    cfg.pick_nums * cfg.sample_from,
)


## 34. Exact repo filtering function

The repository's `KNN_dis_search_distance()` contains a slightly unusual reshape/indexing pattern.

It does:

```python
k_th = k_th_distance.view(length, -1)
...
topk(k_th, num_points, dim=0)
```

where:

```text
length = sample_from = 600
total candidates = pick_nums × length = 1200
```

so the score tensor becomes:

```text
[600, 2]
```

Then it reconstructs flat candidate indices with:

```python
i * length + selected_row
```

We reproduce this **exactly** first.

Later in this section we also compute a logically grouped version for comparison, because the repo's reshape does not obviously preserve the anchor-major candidate grouping.


In [ ]:
def repo_knn_dis_search_distance(
    target,
    kth_distance,
    num_points,
    length,
    depth,
):
    # Exact reshape logic from the repo.
    k_th = kth_distance.view(
        length,
        -1
    )

    # Present in repo but not used afterward.
    target_new = target.view(
        length,
        -1,
        depth
    )

    selected_distances, minD_idx = torch.topk(
        k_th,
        num_points,
        dim=0
    )

    minD_idx = minD_idx.squeeze()

    # Make the one-point case robust for notebook use.
    if minD_idx.ndim == 1:
        minD_idx = minD_idx[:, None]

    point_list = []

    for i in range(minD_idx.shape[1]):
        point_list.append(
            i * length + minD_idx[:, i]
        )

    flat_indices = torch.cat(
        point_list,
        dim=0
    ).long()

    selected_points = target[
        flat_indices.to(target.device)
    ]

    return {
        "selected_points": selected_points,
        "flat_indices": flat_indices,
        "selected_distances_matrix": selected_distances,
        "score_matrix": k_th,
    }


## 35. Apply the actual CIFAR-100 shell-script setting: `ID_points_num = 200`

The shell script passes:

```text
--ID_points_num 200
```

so we first reproduce that execution path.


In [ ]:
ID_POINTS_NUM_REPO_LAUNCH = cfg.id_points_num_shell_script

repo_filtered = repo_knn_dis_search_distance(
    target=candidate_features,
    kth_distance=candidate_kth,
    num_points=ID_POINTS_NUM_REPO_LAUNCH,
    length=cfg.sample_from,
    depth=cfg.penultimate_dim,
)

repo_ood_samples = repo_filtered["selected_points"]

print("ID_points_num:", ID_POINTS_NUM_REPO_LAUNCH)
print("Retained OOD shape:", tuple(repo_ood_samples.shape))

expected_repo_ood = (
    cfg.pick_nums
    * ID_POINTS_NUM_REPO_LAUNCH
)

print("Expected retained count:", expected_repo_ood)

assert repo_ood_samples.shape == (
    expected_repo_ood,
    cfg.penultimate_dim,
)

print(
    "\nRepo shell-script path retains",
    len(repo_ood_samples),
    "synthetic features for this class."
)


## 36. Compare with the parser default (`ID_points_num = 2`)

This is **not** what `train_npos_cifar100.sh` launches, but it explains why reading only the parser can lead to the mistaken conclusion that NPOS retains just four synthetic points per class.


In [ ]:
parser_default_filtered = repo_knn_dis_search_distance(
    target=candidate_features,
    kth_distance=candidate_kth,
    num_points=cfg.id_points_num_parser_default,
    length=cfg.sample_from,
    depth=cfg.penultimate_dim,
)

parser_default_ood = parser_default_filtered[
    "selected_points"
]

print(
    "Parser default ID_points_num:",
    cfg.id_points_num_parser_default
)
print(
    "Retained OOD shape under parser default:",
    tuple(parser_default_ood.shape)
)

print(
    "\nSo parser default would retain",
    len(parser_default_ood),
    "points/class, while the provided shell command retains",
    len(repo_ood_samples),
    "points/class."
)


## 37. Check whether filtering moves toward lower ID density

The retained synthetic samples should, by construction, correspond to relatively large candidate K-th-neighbor scores.

We compare:

```text
all Gaussian proposal scores
vs
scores associated with retained candidates
```

Because the repo uses an unusual reshape/index reconstruction, we calculate the retained candidates' actual k-NN scores again directly rather than assuming the stored score matrix aligns perfectly.


In [ ]:
repo_retained_knn = candidate_knn_distances(
    repo_ood_samples,
    index,
    K=cfg.K,
)

repo_retained_kth = repo_retained_knn[
    "kth_distance"
]

print(
    "All proposal mean K-th distance:",
    float(candidate_kth.mean())
)

print(
    "Retained repo OOD mean K-th distance:",
    float(repo_retained_kth.mean())
)

print(
    "All proposal median:",
    float(candidate_kth.median())
)

print(
    "Retained repo OOD median:",
    float(repo_retained_kth.median())
)


## 38. Diagnose the repo reshape/indexing behavior

Our Gaussian candidate tensor was generated in **anchor-major order**:

```text
anchor 0: candidates 0 ... 599
anchor 1: candidates 600 ... 1199
```

The natural grouped score view would therefore be:

```text
[pick_nums, sample_from] = [2, 600]
```

But the repo reshapes the flat score vector as:

```text
[sample_from, pick_nums] = [600, 2]
```

without transposing the candidate ordering.

This means the score columns used by `topk` do not obviously correspond one-to-one with the anchor blocks later reconstructed using `i * length + row`.

We preserve the exact repo path above, but below we calculate the logically anchor-grouped version as a diagnostic. We do **not** silently substitute it for the repo behavior.


In [ ]:
def grouped_anchor_filter(
    candidates,
    kth_distance,
    pick_nums,
    sample_from,
    num_points,
):
    score_by_anchor = kth_distance.view(
        pick_nums,
        sample_from
    )

    selected_distances, local_idx = torch.topk(
        score_by_anchor,
        num_points,
        dim=1
    )

    offsets = (
        torch.arange(pick_nums)
        .view(-1, 1)
        * sample_from
    )

    flat_idx = (
        local_idx.cpu()
        + offsets
    ).reshape(-1).long()

    selected = candidates[
        flat_idx.to(candidates.device)
    ]

    return {
        "selected_points": selected,
        "flat_indices": flat_idx,
        "selected_distances": selected_distances,
    }


grouped_filtered = grouped_anchor_filter(
    candidates=candidate_features,
    kth_distance=candidate_kth,
    pick_nums=cfg.pick_nums,
    sample_from=cfg.sample_from,
    num_points=ID_POINTS_NUM_REPO_LAUNCH,
)

grouped_ood_samples = grouped_filtered[
    "selected_points"
]

grouped_retained_knn = candidate_knn_distances(
    grouped_ood_samples,
    index,
    K=cfg.K,
)

print(
    "Exact repo retained mean K-th distance:",
    float(repo_retained_kth.mean())
)

print(
    "Anchor-grouped retained mean K-th distance:",
    float(
        grouped_retained_knn[
            "kth_distance"
        ].mean()
    )
)

print(
    "\nBoth outputs contain:",
    len(repo_ood_samples),
    "points"
)


## 39. Visualize proposal scores and retained samples

The histogram compares the full candidate score distribution with the actual retained samples from the exact repo indexing path.


In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    candidate_kth.numpy(),
    bins=40,
    alpha=0.6,
    label="All Gaussian candidates"
)

plt.hist(
    repo_retained_kth.numpy(),
    bins=40,
    alpha=0.6,
    label="Retained by repo path"
)

plt.xlabel(
    f"{cfg.K}-th neighbor squared L2 distance"
)
plt.ylabel("Count")
plt.title(
    f"Class {selected_class}: second k-NN filtering"
)
plt.legend()
plt.show()


## 40. Final OOD tensor for one class

For the provided CIFAR-100 shell command:

```text
select = 200
pick_nums = 2
sample_from = 600
ID_points_num = 200
```

the one-class synthesis path is:

```text
600 ID queue features
    ↓
top 200 boundary candidates
    ↓
randomly choose 2 anchors
    ↓
600 Gaussian proposals per anchor
    ↓
1200 proposals
    ↓
second K=300 k-NN filtering
    ↓
400 retained synthetic OOD features
```

These are the 512-D features that are passed into:

```python
model.mlp(ood_samples)
```

during NPOS training.


In [ ]:
ood_samples_one_class = repo_ood_samples

print(
    "Final one-class synthetic OOD tensor:",
    tuple(ood_samples_one_class.shape)
)

print(
    "Feature dimension:",
    ood_samples_one_class.shape[1]
)

assert ood_samples_one_class.shape[1] == 512


## Second k-NN stage complete

We have now implemented the full NPOS synthesis mechanism for **one CIFAR-100 class**:

```text
ID feature queue
→ first k-NN boundary selection
→ boundary-anchor sampling
→ Gaussian proposals
→ second k-NN low-density filtering
→ retained 512-D synthetic OOD features
```

### Next stage

Next we will scale this from one class to all 100 classes and connect it to the open-set classifier:

```text
real ID penultimate features
    ↓
OOD MLP → target 1

synthetic OOD features
    ↓
OOD MLP → target 0

BCEWithLogitsLoss
    ↓
L_open
```

Then we can combine:

```text
L_total
=
w_disp × L_disp
+
w_comp × L_comp
+
alpha × L_open
```

and perform one complete NPOS training-step sanity check before designing the multi-epoch training loop.


## 41. Scale NPOS synthesis to all 100 classes

The repository loops over every CIFAR-100 class once the class queues are full, generates class-wise synthetic OOD features, and concatenates them.

Before running this section, make sure all 100 queues are full.


In [ ]:
print("All class queues full:", feature_queue.all_classes_full())
print(feature_queue.summary())

if not feature_queue.all_classes_full():
    print(
        "\nRun this first:\n"
        "all_full = fill_all_class_queues(model, train_loader, feature_queue)\n"
        "print(all_full)"
    )


## 42. One-class synthesis helper

This combines the stages already validated:

```text
class queue
→ first KNN boundary selection
→ choose anchors
→ Gaussian proposals
→ second KNN filtering
→ retained 512-D OOD features
```

For the provided CIFAR-100 launch command we use:

```text
K = 300
select = 200
pick_nums = 2
sample_from = 600
ID_points_num = 200
cov_mat = 0.1
```


In [ ]:
def synthesize_ood_for_class(
    class_features,
    cfg,
    seed=None,
):
    class_features_norm = F.normalize(
        class_features,
        dim=1
    )

    class_index = faiss.IndexFlatL2(
        cfg.penultimate_dim
    )

    class_index.add(
        class_features_norm
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    boundary_result = knn_dis_search_decrease(
        target=class_features,
        index=class_index,
        K=cfg.K,
        select=cfg.select,
    )

    boundary_idx = boundary_result[
        "selected_idx"
    ].to(class_features.device)

    boundary_pool = class_features[
        boundary_idx
    ]

    _, anchors = sample_boundary_anchors(
        boundary_pool,
        pick_nums=cfg.pick_nums,
        seed=seed,
    )

    if seed is not None:
        generator = torch.Generator(
            device=class_features.device
        )
        generator.manual_seed(seed)

        negative_samples = torch.randn(
            cfg.sample_from,
            cfg.penultimate_dim,
            generator=generator,
            device=class_features.device,
        )
    else:
        negative_samples = torch.randn(
            cfg.sample_from,
            cfg.penultimate_dim,
            device=class_features.device,
        )

    candidates = generate_gaussian_candidates(
        anchors,
        negative_samples,
        cov_mat=cfg.cov_mat,
    )

    candidate_knn = candidate_knn_distances(
        candidates,
        class_index,
        K=cfg.K,
    )

    filtered = repo_knn_dis_search_distance(
        target=candidates,
        kth_distance=candidate_knn["kth_distance"],
        num_points=cfg.id_points_num_shell_script,
        length=cfg.sample_from,
        depth=cfg.penultimate_dim,
    )

    return filtered["selected_points"]


## 43. Generate OOD features for all classes

With 100 classes, 2 selected anchors, and 200 retained candidates per anchor, the provided launch path produces:

```text
100 × 2 × 200 = 40,000
```

synthetic 512-D OOD features per synthesis step.


In [ ]:
def synthesize_ood_all_classes(
    feature_queue,
    cfg,
    seed=SEED,
    progress_every=10,
):
    all_ood = []

    for cls in range(cfg.num_classes):
        class_features = feature_queue.class_features(cls)

        if len(class_features) < cfg.queue_size:
            raise RuntimeError(
                f"Class {cls} queue is not full: "
                f"{len(class_features)}/{cfg.queue_size}"
            )

        class_ood = synthesize_ood_for_class(
            class_features,
            cfg,
            seed=seed + cls,
        )

        all_ood.append(class_ood)

        if (
            cls == 0
            or (cls + 1) % progress_every == 0
            or cls == cfg.num_classes - 1
        ):
            print(
                f"Class {cls + 1}/{cfg.num_classes}: "
                f"{len(class_ood)} OOD features"
            )

    return torch.cat(
        all_ood,
        dim=0
    )


In [ ]:
# if not feature_queue.all_classes_full():
#     raise RuntimeError(
#         "All 100 queues must be full first. "
#         "Run fill_all_class_queues(...) and then rerun this cell."
#     )

# ood_samples_all = synthesize_ood_all_classes(
#     feature_queue,
#     cfg,
# )

# expected_total_ood = (
#     cfg.num_classes
#     * cfg.pick_nums
#     * cfg.id_points_num_shell_script
# )

# print("\nAll-class OOD tensor:", tuple(ood_samples_all.shape))
# print("Expected count:", expected_total_ood)

# assert ood_samples_all.shape == (
#     expected_total_ood,
#     cfg.penultimate_dim,
# )


## 44. Open-set BCE loss

The repo sends:

```text
real ID 512-D features → OOD MLP → target 1
synthetic OOD features → OOD MLP → target 0
```

and uses `BCEWithLogitsLoss`.


In [ ]:
criterion_bce = nn.BCEWithLogitsLoss()


def compute_open_loss(
    model,
    id_penultimate,
    ood_samples,
):
    id_logits = model.mlp(
        id_penultimate
    ).view(-1)

    ood_logits = model.mlp(
        ood_samples
    ).view(-1)

    logits = torch.cat(
        [id_logits, ood_logits],
        dim=0
    )

    labels = torch.cat(
        [
            torch.ones_like(id_logits),
            torch.zeros_like(ood_logits),
        ],
        dim=0
    )

    open_loss = criterion_bce(
        logits,
        labels
    )

    return open_loss, id_logits, ood_logits


## 45. One complete NPOS training-step sanity check

We now combine:

$$
L_{total}
=
w_{disp}L_{disp}
+
w_{comp}L_{comp}
+
\alpha L_{open}
$$

This mirrors the repo training branch used after the queues are full and `epoch >= start_epoch_KNN`.


In [ ]:
# views, labels = next(iter(train_loader))
# view1, view2 = views

# fullstep_images = torch.cat(
#     [view1, view2],
#     dim=0
# ).to(device)

# fullstep_targets = labels.repeat(2).to(device)

# model.train()

# fullstep_penultimate = model.encoder(
#     fullstep_images
# )

# fullstep_projected = model.head(
#     fullstep_penultimate
# )

# fullstep_features = F.normalize(
#     fullstep_projected,
#     dim=1
# )

# fullstep_comp = comp_criterion(
#     fullstep_features,
#     prototypes,
#     fullstep_targets,
# )

# fullstep_disp, next_prototypes = disp_criterion(
#     fullstep_features,
#     fullstep_targets,
#     prototypes,
# )

# fullstep_open, id_logits, ood_logits = compute_open_loss(
#     model,
#     fullstep_penultimate,
#     ood_samples_all,
# )

# total_npos_loss = (
#     cfg.w_disp * fullstep_disp
#     + cfg.w_comp * fullstep_comp
#     + cfg.loss_weight * fullstep_open
# )

# print("DispLoss: ", float(fullstep_disp.detach().cpu()))
# print("CompLoss: ", float(fullstep_comp.detach().cpu()))
# print("OpenLoss: ", float(fullstep_open.detach().cpu()))
# print("TotalLoss:", float(total_npos_loss.detach().cpu()))

# assert torch.isfinite(total_npos_loss)


## 46. Inspect ID and OOD logits

The OOD MLP has barely been trained, so strong separation is not expected yet. This is only a sanity check.


In [ ]:
# print(
#     "ID logits mean/std:",
#     float(id_logits.detach().mean().cpu()),
#     float(id_logits.detach().std().cpu())
# )

# print(
#     "OOD logits mean/std:",
#     float(ood_logits.detach().mean().cpu()),
#     float(ood_logits.detach().std().cpu())
# )

# print("\nTargets: ID = 1, synthetic OOD = 0")


## 47. Full backward pass

Expected:

```text
encoder → gradients
projection head → gradients
OOD MLP → gradients
fc → still no gradient from the shown total loss
```


In [ ]:
# optimizer.zero_grad()

# total_npos_loss.backward()

# print("Encoder gradients:", grad_summary(model.encoder))
# print("Head gradients:   ", grad_summary(model.head))
# print("FC gradients:     ", grad_summary(model.fc))
# print("OOD MLP gradients:", grad_summary(model.mlp))

# optimizer.step()

# prototypes = next_prototypes.detach()

# print("\nComplete NPOS backward + optimizer step succeeded.")


## Complete NPOS training step validated

The notebook now covers:

```text
CIFAR-100
→ ResNet-34
→ closed-set prototype learning
→ 512-D class queues
→ first KNN boundary selection
→ Gaussian proposals
→ second KNN filtering
→ all-class synthetic OOD
→ OOD MLP + BCE
→ full total loss
→ backward + optimizer step
```

### Next stage

Next we should build the real multi-epoch training loop with:

```text
epochs 0–199:
    queue maintenance
    DispLoss + CompLoss

epochs 200–499:
    queue maintenance
    all-class NPOS synthesis
    DispLoss + CompLoss + OpenLoss
```

Before launching it on Kaggle, we should also add checkpointing and resume support so a session interruption does not lose the run.


## 49. Fidelity correction before long training

Before launching 500 epochs, we need to correct one detail in the earlier notebook implementation of `DispLoss`.

The official repo updates the prototype tensor with assignments from the **non-detached** projected features, then computes the dispersion loss from that locally updated tensor. In PyTorch, those indexed assignments create a `CopySlices` autograd path, so `DispLoss` can contribute gradients back to the current batch features.

Our earlier diagnostic implementation used `torch.no_grad()` and `features.detach()` during the prototype update. That was useful for inspection, but it removed the `DispLoss → feature` gradient path.

The training loop below therefore uses a repo-faithful implementation. The persistent prototype state is detached **after** the local differentiable update, matching the repository logic.


In [ ]:
class RepoFaithfulDispLoss(nn.Module):
    def __init__(
        self,
        num_classes,
        feat_dim,
        proto_m=0.95,
        temperature=0.1,
        base_temperature=0.1,
    ):
        super().__init__()

        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.proto_m = proto_m
        self.temperature = temperature
        self.base_temperature = base_temperature

    def forward(self, features, labels, prototypes):
        # Start from detached persistent state.
        # Do NOT use no_grad for the local updates: this preserves
        # the repo's CopySlices autograd path from current features.
        local_prototypes = prototypes.detach().clone()

        for j in range(len(features)):
            cls = int(labels[j].item())

            local_prototypes[cls] = F.normalize(
                local_prototypes[cls] * self.proto_m
                + features[j] * (1.0 - self.proto_m),
                dim=0,
            )

        labels_proto = torch.arange(
            self.num_classes,
            device=features.device,
        ).view(-1, 1)

        mask = (
            1.0
            - torch.eq(
                labels_proto,
                labels_proto.T,
            ).float()
        )

        logits = (
            local_prototypes
            @ local_prototypes.T
        ) / self.temperature

        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(
                self.num_classes,
                device=features.device,
            ).view(-1, 1),
            0,
        )

        mask = mask * logits_mask

        mean_prob_neg = torch.log(
            (mask * torch.exp(logits)).sum(dim=1)
            / mask.sum(dim=1)
        )

        mean_prob_neg = mean_prob_neg[
            ~torch.isnan(mean_prob_neg)
        ]

        loss = (
            self.temperature
            / self.base_temperature
        ) * mean_prob_neg.mean()

        # Persistent state for the next iteration is detached.
        next_prototypes = local_prototypes.detach()

        return loss, next_prototypes


train_disp_criterion = RepoFaithfulDispLoss(
    num_classes=cfg.num_classes,
    feat_dim=cfg.feat_dim,
    proto_m=cfg.proto_m,
    temperature=cfg.temperature,
    base_temperature=cfg.temperature,
)

print("Repo-faithful DispLoss ready.")


## 50. Verify that `DispLoss` now contributes feature gradients

This tiny check isolates the dispersion term. We expect a non-zero gradient on at least some current-batch projected features.


In [ ]:
test_features = torch.randn(
    16,
    cfg.feat_dim,
    device=device,
    requires_grad=True,
)

test_features = F.normalize(
    test_features,
    dim=1,
)

test_labels = torch.arange(
    16,
    device=device,
) % cfg.num_classes

test_proto = prototypes.detach().clone()

test_disp, _ = train_disp_criterion(
    test_features,
    test_labels,
    test_proto,
)

test_disp.backward()

grad_ok = (
    test_features.grad is not None
    if test_features.is_leaf
    else True
)

print("DispLoss requires grad:", test_disp.requires_grad)
print("DispLoss value:", float(test_disp.detach().cpu()))
print(
    "Autograd graph present:",
    test_disp.grad_fn is not None
)

assert test_disp.requires_grad
assert test_disp.grad_fn is not None

print("DispLoss autograd fidelity check passed.")


## 51. Training/checkpoint configuration

The paper experiment runs for 500 epochs with NPOS activated at epoch 200.

The checkpoint contains:

```text
model
optimizer
epoch
prototypes
class feature queue
queue counters
training history
random-number-generator states
```

Checkpoints are written to `/kaggle/working/npos_checkpoints`.

**Kaggle note:** `/kaggle/working` survives during the active session. To resume after the session itself is destroyed, save/commit the notebook output or otherwise persist the checkpoint files as a Kaggle output/dataset.


In [ ]:
from pathlib import Path
import random

# Read the old checkpoint from Kaggle input
RESUME_CHECKPOINT_DIR = Path(
    "/kaggle/input/datasets/yonae1/checkpoint-epoch479"
)

# Save all new checkpoints here
CHECKPOINT_DIR = Path(
    "/kaggle/working/npos_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_LOG_PATH = (
    CHECKPOINT_DIR
    / "training_history.pt"
)

# print("Resume checkpoint directory:", RESUME_CHECKPOINT_DIR)
print("New checkpoint directory:", CHECKPOINT_DIR)

## 52. Fresh optimizer for the actual training run

The official training code uses SGD with Nesterov momentum. The OOD MLP receives one-tenth of the main learning rate:

```text
encoder / fc / projection head: 0.5
OOD MLP:                        0.05
```

We create a **fresh optimizer** here so the earlier sanity-check steps do not affect the real run.


In [ ]:
def build_optimizer(model, cfg):
    return torch.optim.SGD(
        [
            {
                "params": model.encoder.parameters(),
                "name": "encoder",
            },
            {
                "params": model.fc.parameters(),
                "name": "fc",
            },
            {
                "params": model.head.parameters(),
                "name": "head",
            },
            {
                "params": model.mlp.parameters(),
                "lr": cfg.learning_rate * 0.1,
                "name": "mlp",
            },
        ],
        lr=cfg.learning_rate,
        momentum=cfg.momentum,
        nesterov=True,
        weight_decay=cfg.weight_decay,
    )


train_optimizer = build_optimizer(
    model,
    cfg,
)

print(
    "Optimizer group LRs:",
    [
        group["lr"]
        for group in train_optimizer.param_groups
    ]
)


## 53. Cosine learning-rate schedule

The paper specifies cosine decay. We preserve the 10× lower learning rate for the OOD MLP throughout training.


In [ ]:
def set_cosine_lr(
    optimizer,
    epoch,
    total_epochs,
    base_lr,
):
    cosine_factor = 0.5 * (
        1.0
        + math.cos(
            math.pi
            * epoch
            / total_epochs
        )
    )

    main_lr = (
        base_lr
        * cosine_factor
    )

    for group in optimizer.param_groups:
        if group.get("name") == "mlp":
            group["lr"] = (
                main_lr * 0.1
            )
        else:
            group["lr"] = main_lr

    return main_lr


## 54. Checkpoint save/load helpers

Resume restores the queue as well as the network. This matters because NPOS synthesis depends on the class-wise feature memory.


In [ ]:
def save_training_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    prototypes,
    feature_queue,
    history,
):
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "prototypes": prototypes.detach().cpu(),
        "queue_data": feature_queue.data.detach().cpu(),
        "queue_counts": feature_queue.counts.detach().cpu(),
        "history": history,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
    }

    if torch.cuda.is_available():
        state[
            "cuda_rng_state_all"
        ] = torch.cuda.get_rng_state_all()

    torch.save(
        state,
        path,
    )


def load_training_checkpoint(
    path,
    model,
    optimizer,
    feature_queue,
    device,
):
    checkpoint = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    prototypes_loaded = (
        checkpoint["prototypes"]
        .to(device)
    )

    feature_queue.data.copy_(
        checkpoint["queue_data"].to(device)
    )

    feature_queue.counts.copy_(
        checkpoint["queue_counts"].to(device)
    )

    # ---- Restore CPU PyTorch RNG safely ----
    torch_rng_state = checkpoint["torch_rng_state"]

    if not isinstance(
        torch_rng_state,
        torch.Tensor,
    ):
        torch_rng_state = torch.tensor(
            torch_rng_state,
            dtype=torch.uint8,
        )
    else:
        torch_rng_state = torch_rng_state.to(
            device="cpu",
            dtype=torch.uint8,
        )

    torch.set_rng_state(
        torch_rng_state
    )

    # ---- Restore NumPy RNG ----
    np.random.set_state(
        checkpoint["numpy_rng_state"]
    )

    # ---- Restore Python RNG ----
    random.setstate(
        checkpoint["python_rng_state"]
    )

    # ---- Restore CUDA RNG safely ----
    if (
        torch.cuda.is_available()
        and "cuda_rng_state_all"
        in checkpoint
    ):
        cuda_states = []

        for state in checkpoint[
            "cuda_rng_state_all"
        ]:
            if not isinstance(
                state,
                torch.Tensor,
            ):
                state = torch.tensor(
                    state,
                    dtype=torch.uint8,
                )
            else:
                state = state.to(
                    device="cpu",
                    dtype=torch.uint8,
                )

            cuda_states.append(state)

        torch.cuda.set_rng_state_all(
            cuda_states
        )

    start_epoch = (
        int(checkpoint["epoch"])
        + 1
    )

    history = checkpoint.get(
        "history",
        [],
    )

    return (
        start_epoch,
        prototypes_loaded,
        history,
    )


## 55. Repo-style queue update used during training

The queue is filled until each class reaches capacity. After it is full, new features are inserted FIFO.


In [ ]:
@torch.no_grad()
def update_training_queue(
    feature_queue,
    penultimate,
    targets,
):
    feature_queue.update(
        penultimate.detach(),
        targets,
    )


## 56. One epoch of NPOS training

The two phases are:

```text
epoch < 200:
    update queues
    DispLoss + CompLoss

epoch >= 200:
    update queues
    synthesize OOD for all classes
    DispLoss + CompLoss + OpenLoss
```

For repo fidelity, NPOS synthesis is performed **per training iteration** after activation and after the queues are full.

This is computationally expensive, especially with CPU FAISS in Kaggle. That cost reflects the original algorithmic path rather than an optimized reimplementation.


In [ ]:
def train_one_epoch_npos(
    epoch,
    model,
    train_loader,
    optimizer,
    prototypes,
    feature_queue,
    cfg,
    print_every=20,
):
    model.train()

    running = {
        "total": 0.0,
        "disp": 0.0,
        "comp": 0.0,
        "open": 0.0,
        "acc": 0.0,
        "steps": 0,
    }

    for step, (views, targets) in enumerate(
        train_loader
    ):
        view1, view2 = views

        images = torch.cat(
            [view1, view2],
            dim=0,
        ).to(device)

        targets = targets.repeat(2).to(
            device
        )

        penultimate = model.encoder(
            images
        )

        projected = model.head(
            penultimate
        )

        features = F.normalize(
            projected,
            dim=1,
        )

        # Update the 512-D class memory every iteration.
        update_training_queue(
            feature_queue,
            penultimate,
            targets,
        )

        disp_loss, next_prototypes = (
            train_disp_criterion(
                features,
                targets,
                prototypes,
            )
        )

        comp_loss = comp_criterion(
            features,
            next_prototypes,
            targets,
        )

        open_loss = torch.zeros(
            (),
            device=device,
        )

        npos_active = (
            epoch >= cfg.start_epoch_knn
            and feature_queue.all_classes_full()
        )

        if npos_active:
            # Feature-space synthesis does not require gradients
            # through the generated samples.
            with torch.no_grad():
                ood_samples = (
                    synthesize_ood_all_classes(
                        feature_queue,
                        cfg,
                        seed=(
                            SEED
                            + epoch * len(train_loader)
                            + step
                        ),
                        progress_every=1000,
                    )
                )

            open_loss, _, _ = (
                compute_open_loss(
                    model,
                    penultimate,
                    ood_samples,
                )
            )

        total_loss = (
            cfg.w_disp * disp_loss
            + cfg.w_comp * comp_loss
            + cfg.loss_weight * open_loss
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        total_loss.backward()

        optimizer.step()

        # Persistent prototype state moves to next iteration.
        prototypes = (
            next_prototypes.detach()
        )

        with torch.no_grad():
            logits = model.fc(
                penultimate
            )

            pred = logits.argmax(
                dim=1
            )

            batch_acc = (
                pred.eq(targets)
                .float()
                .mean()
                .item()
            )

        running["total"] += float(
            total_loss.detach().cpu()
        )
        running["disp"] += float(
            disp_loss.detach().cpu()
        )
        running["comp"] += float(
            comp_loss.detach().cpu()
        )
        running["open"] += float(
            open_loss.detach().cpu()
        )
        running["acc"] += batch_acc
        running["steps"] += 1

        if (
            step % print_every == 0
            or step == len(train_loader) - 1
        ):
            print(
                f"Epoch {epoch:03d} "
                f"[{step:03d}/{len(train_loader):03d}] "
                f"total={float(total_loss.detach()):.4f} "
                f"disp={float(disp_loss.detach()):.4f} "
                f"comp={float(comp_loss.detach()):.4f} "
                f"open={float(open_loss.detach()):.4f} "
                f"acc={batch_acc*100:.2f}% "
                f"queue_full={feature_queue.all_classes_full()} "
                f"npos={npos_active}"
            )

    n = running["steps"]

    epoch_metrics = {
        "epoch": epoch,
        "total_loss": running["total"] / n,
        "disp_loss": running["disp"] / n,
        "comp_loss": running["comp"] / n,
        "open_loss": running["open"] / n,
        "train_acc": running["acc"] / n,
    }

    return (
        epoch_metrics,
        prototypes,
    )


## 57. Evaluation helper

The raw repository validation function appears to call `model(input)` and treat the 128-D normalized projection as 100-class logits. That is inconsistent with the training accuracy path, which uses `model.fc(penultimate)`.

For the actual notebook run, we report ID classification accuracy using the explicit classifier branch:

```text
encoder → 512-D → fc → 100 logits
```

This is a documented notebook choice rather than a silent claim that the raw validation function is correct.


In [ ]:
@torch.no_grad()
def evaluate_id_accuracy(
    model,
    loader,
):
    model.eval()

    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        penultimate = model.encoder(
            images
        )

        logits = model.fc(
            penultimate
        )

        pred = logits.argmax(
            dim=1
        )

        correct += int(
            pred.eq(targets).sum()
        )
        total += len(targets)

    return correct / total


## 58. Full 500-epoch training driver with resume support

This cell **defines** the training driver. It does not automatically start 500 epochs.

Use the launch cell afterward when you are ready.


In [ ]:
def run_training(
    model,
    train_loader,
    test_loader,
    cfg,
    feature_queue,
    optimizer,
    prototypes,
    start_epoch=0,
    history=None,
    checkpoint_every=1,
):
    if history is None:
        history = []

    for epoch in range(
        start_epoch,
        cfg.epochs,
    ):
        main_lr = set_cosine_lr(
            optimizer,
            epoch,
            cfg.epochs,
            cfg.learning_rate,
        )

        print(
            "\\n"
            + "=" * 72
        )
        print(
            f"Epoch {epoch}/{cfg.epochs - 1} "
            f"| main LR={main_lr:.6f} "
            f"| NPOS active="
            f"{epoch >= cfg.start_epoch_knn}"
        )
        print("=" * 72)

        metrics, prototypes = (
            train_one_epoch_npos(
                epoch=epoch,
                model=model,
                train_loader=train_loader,
                optimizer=optimizer,
                prototypes=prototypes,
                feature_queue=feature_queue,
                cfg=cfg,
            )
        )

        id_acc = evaluate_id_accuracy(
            model,
            test_loader,
        )

        metrics["id_test_acc"] = id_acc
        metrics["main_lr"] = main_lr
        metrics["queue_full"] = (
            feature_queue.all_classes_full()
        )

        history.append(
            metrics
        )

        print(
            f"Epoch {epoch} summary | "
            f"total={metrics['total_loss']:.4f} | "
            f"disp={metrics['disp_loss']:.4f} | "
            f"comp={metrics['comp_loss']:.4f} | "
            f"open={metrics['open_loss']:.4f} | "
            f"train_acc={metrics['train_acc']*100:.2f}% | "
            f"ID test acc={id_acc*100:.2f}%"
        )

        if (
            (epoch + 1) % checkpoint_every == 0
            or epoch == cfg.epochs - 1
        ):
            ckpt_path = (
                CHECKPOINT_DIR
                / f"epoch_{epoch:03d}.pt"
            )

            save_training_checkpoint(
                path=ckpt_path,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                prototypes=prototypes,
                feature_queue=feature_queue,
                history=history,
            )

            latest_path = (
                CHECKPOINT_DIR
                / "latest.pt"
            )

         

            save_training_checkpoint(
                path=latest_path,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                prototypes=prototypes,
                feature_queue=feature_queue,
                history=history,
            )

            torch.save(
                history,
                TRAIN_LOG_PATH,
            )

            print(
                "Saved checkpoint:",
                latest_path,
            )

    return prototypes, history


## 59. Recommended dry run before 500 epochs

Before committing to the full run, execute one epoch with NPOS disabled. This checks the real epoch loop and checkpoint code without paying the post-epoch-200 synthesis cost.

This cell temporarily runs **one epoch only** and then restores `cfg.epochs`.


In [ ]:
# Optional but strongly recommended.
#
# original_epochs = cfg.epochs
# cfg.epochs = 1
#
# dry_optimizer = build_optimizer(model, cfg)
# dry_history = []
#
# prototypes, dry_history = run_training(
#     model=model,
#     train_loader=train_loader,
#     test_loader=test_loader,
#     cfg=cfg,
#     feature_queue=feature_queue,
#     optimizer=dry_optimizer,
#     prototypes=prototypes,
#     start_epoch=0,
#     history=dry_history,
#     checkpoint_every=1,
# )
#
# cfg.epochs = original_epochs
#
# print(dry_history[-1])


## 60. Start a fresh 500-epoch run

Uncomment this cell only when you are ready to train.

Because the notebook has already executed sanity-check optimizer steps, for the cleanest experimental run you should restart the Kaggle kernel, rerun the notebook definitions/data/model initialization, **skip the diagnostic optimizer-step cells**, and then launch from here.


In [ ]:
# # FRESH RUN

# train_optimizer = build_optimizer(
#     model,
#     cfg,
# )

# training_history = []
# start_epoch = 0

# prototypes, training_history = run_training(
#     model=model,
#     train_loader=train_loader,
#     test_loader=test_loader,
#     cfg=cfg,
#     feature_queue=feature_queue,
#     optimizer=train_optimizer,
#     prototypes=prototypes,
#     start_epoch=start_epoch,
#     history=training_history,
#     checkpoint_every=80,
# )


## 61. Resume from the latest checkpoint

Use this instead of the fresh-run cell after restoring `latest.pt` into `/kaggle/working/npos_checkpoints/`.


In [ ]:
# RESUME RUN

latest_path = (
    CHECKPOINT_DIR
    / "epoch_491.pt"
)

train_optimizer = build_optimizer(
    model,
    cfg,
)

(
    start_epoch,
    prototypes,
    training_history,
) = load_training_checkpoint(
    path=latest_path,
    model=model,
    optimizer=train_optimizer,
    feature_queue=feature_queue,
    device=device,
)

print(
    "Resuming at epoch:",
    start_epoch
)
print(
    "Queue summary:",
    feature_queue.summary()
)

prototypes, training_history = run_training(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    cfg=cfg,
    feature_queue=feature_queue,
    optimizer=train_optimizer,
    prototypes=prototypes,
    start_epoch=start_epoch,
    history=training_history,
    checkpoint_every=4,
)


## 62. Plot training history

After training (or after resuming), this cell plots the main tracked quantities.


In [ ]:
def plot_history(history):
    if len(history) == 0:
        print("No training history yet.")
        return

    epochs = [
        row["epoch"]
        for row in history
    ]

    keys = [
        "total_loss",
        "disp_loss",
        "comp_loss",
        "open_loss",
        "train_acc",
        "id_test_acc",
    ]

    for key in keys:
        values = [
            row[key]
            for row in history
        ]

        plt.figure(figsize=(8, 4))
        plt.plot(
            epochs,
            values,
        )
        plt.xlabel("Epoch")
        plt.ylabel(key)
        plt.title(key)
        plt.show()


# After training:
# plot_history(training_history)


## Training notebook stage complete

The notebook now contains the complete experiment path:

```text
data preparation
→ repo-style CIFAR ResNet-34
→ prototype losses
→ 512-D queues
→ two-stage NPOS synthesis
→ OOD MLP/BCE
→ repo-faithful DispLoss gradients
→ cosine LR
→ 500-epoch training loop
→ ID accuracy evaluation
→ checkpoint/resume
→ training-history plots
```

The next major stage after training is **OOD evaluation** on SVHN, Places365, LSUN, iSUN, and Texture, with FPR95/AUROC comparison against the paper's reported CIFAR-100 results.


## v9: Kaggle T4 ×2 DataParallel

This version automatically uses both Kaggle T4 GPUs when available.

Wrapped modules:
- encoder
- projection head
- classifier `fc`
- OOD `mlp`

Build the optimizer only after the wrappers are applied. The existing Fresh Run and Resume Run cells already do that.

For a clean experiment, restart the Kaggle kernel, rerun the notebook from the beginning, confirm `GPU count: 2` and that the wrappers print `DataParallel`, then use **FRESH RUN**.

Use **RESUME RUN** only with checkpoints created by this DataParallel version unless checkpoint key conversion is handled explicitly.


In [9]:
# ============================================================
# Restart / Evaluation Setup
# Run this AFTER the model architecture cell
# ============================================================

import torch
import torch.nn.functional as F
import faiss
import numpy as np

from torchvision import datasets
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)


# ------------------------------------------------------------
# 2. Load checkpoint
# ------------------------------------------------------------

ckpt_path = "/kaggle/working/npos_checkpoints/epoch_499.pt"

ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint epoch:", ckpt["epoch"])


# ------------------------------------------------------------
# 3. Load trained weights into model
# Assumes 'model' was already created in the previous cell
# ------------------------------------------------------------

model.load_state_dict(
    ckpt["model_state_dict"]
)

model = model.to(DEVICE)
model.eval()

print("Model weights loaded successfully")


# ------------------------------------------------------------
# 4. Create deterministic CIFAR-100 evaluation datasets
# ------------------------------------------------------------

train_eval_dataset = datasets.CIFAR100(
    root=cfg.data_root,
    train=True,
    download=False,
    transform=test_transform,
)

test_eval_dataset = datasets.CIFAR100(
    root=cfg.data_root,
    train=False,
    download=False,
    transform=test_transform,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

print("CIFAR-100 train:", len(train_eval_dataset))
print("CIFAR-100 test :", len(test_eval_dataset))


# ------------------------------------------------------------
# 5. Feature extraction function
# ------------------------------------------------------------

@torch.no_grad()
def extract_features(model, loader, device):

    model.eval()
    features = []

    for images, _ in tqdm(loader):

        images = images.to(device)

        # 512-d penultimate representation
        h = model.encoder(images)

        # L2 normalization
        h = F.normalize(h, dim=1)

        features.append(h.cpu())

    return torch.cat(features, dim=0)


# ------------------------------------------------------------
# 6. Extract CIFAR-100 reference/test features
# ------------------------------------------------------------

print("\nExtracting CIFAR-100 train features...")

ftrain = extract_features(
    model,
    train_eval_loader,
    DEVICE
)

print("\nExtracting CIFAR-100 test features...")

ftest = extract_features(
    model,
    test_eval_loader,
    DEVICE
)

print("\nTrain features:", ftrain.shape)
print("Test features :", ftest.shape)

print(
    "Train feature norm:",
    ftrain[0].norm()
)

print(
    "Test feature norm:",
    ftest[0].norm()
)


# ------------------------------------------------------------
# 7. Build FAISS KNN reference index
# ------------------------------------------------------------

K = 300

index = faiss.IndexFlatL2(
    ftrain.shape[1]
)

index.add(
    ftrain.numpy()
)

print(
    "\nFAISS reference samples:",
    index.ntotal
)


# ------------------------------------------------------------
# 8. Compute CIFAR-100 ID scores
# ------------------------------------------------------------

D_in, _ = index.search(
    ftest.numpy(),
    K
)

scores_in = -D_in[:, -1]

print(
    "ID scores shape:",
    scores_in.shape
)

print(
    "Mean ID score:",
    scores_in.mean()
)

print("\nEvaluation setup complete.")

Using device: cuda
Checkpoint epoch: 499
Model weights loaded successfully
CIFAR-100 train: 50000
CIFAR-100 test : 10000

Extracting CIFAR-100 train features...


  0%|          | 0/196 [00:00<?, ?it/s]


Extracting CIFAR-100 test features...


  0%|          | 0/40 [00:00<?, ?it/s]


Train features: torch.Size([50000, 512])
Test features : torch.Size([10000, 512])
Train feature norm: tensor(1.)
Test feature norm: tensor(1.0000)

FAISS reference samples: 50000
ID scores shape: (10000,)
Mean ID score: -0.25042668

Evaluation setup complete.


In [10]:
import numpy as np
import sklearn.metrics as sk

def stable_cumsum(arr, rtol=1e-05, atol=1e-08):
    out = np.cumsum(arr, dtype=np.float64)
    expected = np.sum(arr, dtype=np.float64)

    if not np.allclose(out[-1], expected, rtol=rtol, atol=atol):
        raise RuntimeError("cumsum was found to be unstable")

    return out


def fpr_and_fdr_at_recall(
    y_true,
    y_score,
    recall_level=0.95,
    pos_label=1
):
    y_true = (y_true == pos_label)

    desc_score_indices = np.argsort(
        y_score,
        kind="mergesort"
    )[::-1]

    y_score = y_score[desc_score_indices]
    y_true = y_true[desc_score_indices]

    distinct_value_indices = np.where(
        np.diff(y_score)
    )[0]

    threshold_idxs = np.r_[
        distinct_value_indices,
        y_true.size - 1
    ]

    tps = stable_cumsum(
        y_true
    )[threshold_idxs]

    fps = 1 + threshold_idxs - tps

    recall = tps / tps[-1]

    last_ind = tps.searchsorted(tps[-1])

    sl = slice(
        last_ind,
        None,
        -1
    )

    recall = np.r_[
        recall[sl],
        1
    ]

    fps = np.r_[
        fps[sl],
        0
    ]

    cutoff = np.argmin(
        np.abs(
            recall - recall_level
        )
    )

    return fps[cutoff] / np.sum(
        np.logical_not(y_true)
    )


def get_measures(
    pos,
    neg,
    recall_level=0.95
):
    pos = np.asarray(pos)
    neg = np.asarray(neg)

    examples = np.concatenate(
        (pos, neg)
    )

    labels = np.zeros(
        len(examples),
        dtype=np.int32
    )

    labels[:len(pos)] = 1

    auroc = sk.roc_auc_score(
        labels,
        examples
    )

    aupr = sk.average_precision_score(
        labels,
        examples
    )

    fpr = fpr_and_fdr_at_recall(
        labels,
        examples,
        recall_level=recall_level,
        pos_label=1
    )

    return auroc, aupr, fpr

In [25]:
# ============================================================
# Download official SVHN test set used by original NPOS
# ============================================================

!mkdir -p /kaggle/working/svhn

!wget -O /kaggle/working/svhn/test_32x32.mat \
http://ufldl.stanford.edu/housenumbers/test_32x32.mat

import os

svhn_path = "/kaggle/working/svhn/test_32x32.mat"

print("Exists:", os.path.exists(svhn_path))
print("Size MB:", os.path.getsize(svhn_path) / (1024**2))

# ============================================================
# SVHN OOD Evaluation — official .mat / NPOS-style
# ============================================================

import scipy.io as sio
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Official SVHN test file
# ------------------------------------------------------------

SVHN_PATH = "/kaggle/working/svhn/test_32x32.mat"


# ------------------------------------------------------------
# 2. Load .mat exactly in the same style as NPOS svhn_loader.py
# ------------------------------------------------------------

loaded_mat = sio.loadmat(SVHN_PATH)

svhn_data = loaded_mat["X"]   # [32, 32, 3, N]
svhn_labels = loaded_mat["y"]

# label 10 corresponds to digit 0
svhn_labels = (svhn_labels % 10).squeeze()

# Original NPOS loader transposes to [N, 3, 32, 32]
svhn_data = np.transpose(
    svhn_data,
    (3, 2, 0, 1)
)

print("SVHN data shape:", svhn_data.shape)
print("SVHN labels shape:", svhn_labels.shape)


# ------------------------------------------------------------
# 3. NPOS CIFAR-100 normalization
# ------------------------------------------------------------

svhn_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5071, 0.4867, 0.4408),
        (0.2675, 0.2565, 0.2761)
    ),
])


# ------------------------------------------------------------
# 4. Dataset wrapper
# ------------------------------------------------------------

class SVHNMatDataset(Dataset):
    def __init__(self, data, labels, transform=None):
        self.data = data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        # [3, 32, 32] -> [32, 32, 3]
        img = np.transpose(
            self.data[idx],
            (1, 2, 0)
        )

        img = Image.fromarray(img)

        if self.transform is not None:
            img = self.transform(img)

        return img, int(self.labels[idx])


svhn_dataset = SVHNMatDataset(
    svhn_data,
    svhn_labels,
    transform=svhn_transform
)

print("SVHN samples:", len(svhn_dataset))


# ------------------------------------------------------------
# 5. Match original NPOS OOD sample cap
# Randomly select 10,000 if dataset is larger
# ------------------------------------------------------------

rng = np.random.default_rng(42)

if len(svhn_dataset) > 10000:

    indices = rng.choice(
        len(svhn_dataset),
        size=10000,
        replace=False
    )

    svhn_dataset = torch.utils.data.Subset(
        svhn_dataset,
        indices
    )

print("SVHN evaluation samples:", len(svhn_dataset))


# ------------------------------------------------------------
# 6. DataLoader
# ------------------------------------------------------------

svhn_loader = DataLoader(
    svhn_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


# ------------------------------------------------------------
# 7. Extract normalized 512-d features
# ------------------------------------------------------------

@torch.no_grad()
def extract_ood_features(model, loader, device):

    model.eval()
    features = []

    for images, _ in tqdm(loader):

        images = images.to(device)

        h = model.encoder(images)
        h = F.normalize(h, dim=1)

        features.append(h.cpu())

    return torch.cat(features, dim=0)


fsvhn = extract_ood_features(
    model,
    svhn_loader,
    DEVICE
)

print("SVHN feature shape:", fsvhn.shape)
print("Feature norm:", fsvhn[0].norm())


# ------------------------------------------------------------
# 8. KNN scoring
# ------------------------------------------------------------

K = 300

D_svhn, _ = index.search(
    fsvhn.numpy(),
    K
)

scores_svhn = -D_svhn[:, -1]

print("\nMean ID score   :", scores_in.mean())
print("Mean SVHN score :", scores_svhn.mean())


# ------------------------------------------------------------
# 9. Original-style NPOS metrics
# ------------------------------------------------------------

auroc_svhn, aupr_svhn, fpr95_svhn = get_measures(
    scores_in,
    scores_svhn
)

print("\n========== SVHN OOD RESULTS ==========")
print(f"AUROC : {auroc_svhn * 100:.2f}")
print(f"AUPR  : {aupr_svhn * 100:.2f}")
print(f"FPR95 : {fpr95_svhn * 100:.2f}")
print("======================================")

--2026-09-13 15:20:42--  http://ufldl.stanford.edu/housenumbers/test_32x32.mat
Resolving ufldl.stanford.edu (ufldl.stanford.edu)... 171.64.68.10
Connecting to ufldl.stanford.edu (ufldl.stanford.edu)|171.64.68.10|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64275384 (61M) [text/plain]
Saving to: ‘/kaggle/working/svhn/test_32x32.mat’

/kaggle/working/svh 100%[===================>]  61.30M  16.1MB/s    in 4.6s    

2026-09-13 15:20:47 (13.2 MB/s) - ‘/kaggle/working/svhn/test_32x32.mat’ saved [64275384/64275384]

Exists: True
Size MB: 61.29778289794922
SVHN data shape: (26032, 3, 32, 32)
SVHN labels shape: (26032,)
SVHN samples: 26032
SVHN evaluation samples: 10000


  0%|          | 0/40 [00:00<?, ?it/s]

SVHN feature shape: torch.Size([10000, 512])
Feature norm: tensor(1.0000)

Mean ID score   : -0.25042668
Mean SVHN score : -0.68082273

========== SVHN OOD RESULTS ==========
AUROC : 97.29
AUPR  : 97.14
FPR95 : 12.93


In [26]:
# ============================================================
# Textures (DTD) OOD Evaluation — official DTD / NPOS-style
# ============================================================

# Download + extract official DTD
!wget -O /kaggle/working/dtd-r1.0.1.tar.gz \
https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz

!tar -xvzf /kaggle/working/dtd-r1.0.1.tar.gz \
-C /kaggle/working/


import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. DTD root
# ------------------------------------------------------------

DTD_ROOT = "/kaggle/working/dtd/images"


# ------------------------------------------------------------
# 2. Original-style texture transform
# Resize + CenterCrop to 32x32, then CIFAR-100 normalization
# ------------------------------------------------------------

texture_transform = transforms.Compose([
    transforms.Resize(32),
    transforms.CenterCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5071, 0.4867, 0.4408),
        (0.2675, 0.2565, 0.2761)
    ),
])


# ------------------------------------------------------------
# 3. Load full DTD images folder
# ------------------------------------------------------------

textures_dataset = datasets.ImageFolder(
    root=DTD_ROOT,
    transform=texture_transform
)

print("Original DTD samples:", len(textures_dataset))


# ------------------------------------------------------------
# 4. Match original OOD evaluation style:
# randomly sample up to 10,000 images
# ------------------------------------------------------------

rng = np.random.default_rng(42)

if len(textures_dataset) > 10000:
    indices = rng.choice(
        len(textures_dataset),
        size=10000,
        replace=False
    )
    textures_dataset = torch.utils.data.Subset(
        textures_dataset,
        indices
    )

print("Textures evaluation samples:", len(textures_dataset))


# ------------------------------------------------------------
# 5. DataLoader
# ------------------------------------------------------------

textures_loader = DataLoader(
    textures_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


# ------------------------------------------------------------
# 6. Extract normalized 512-d features
# ------------------------------------------------------------

@torch.no_grad()
def extract_ood_features(model, loader, device):
    model.eval()
    features = []

    for images, _ in tqdm(loader):
        images = images.to(device)

        h = model.encoder(images)
        h = F.normalize(h, dim=1)

        features.append(h.cpu())

    return torch.cat(features, dim=0)


ftextures = extract_ood_features(
    model,
    textures_loader,
    DEVICE
)

print("Textures feature shape:", ftextures.shape)
print("Feature norm:", ftextures[0].norm())


# ------------------------------------------------------------
# 7. KNN scoring
# ------------------------------------------------------------

K = 300

D_tex, _ = index.search(
    ftextures.numpy(),
    K
)

scores_textures = -D_tex[:, -1]

print("\nMean ID score       :", scores_in.mean())
print("Mean Textures score :", scores_textures.mean())


# ------------------------------------------------------------
# 8. OOD metrics
# ------------------------------------------------------------

auroc_tex, aupr_tex, fpr95_tex = get_measures(
    scores_in,
    scores_textures
)

print("\n========== TEXTURES OOD RESULTS ==========")
print(f"AUROC : {auroc_tex * 100:.2f}")
print(f"AUPR  : {aupr_tex * 100:.2f}")
print(f"FPR95 : {fpr95_tex * 100:.2f}")
print("==========================================")

--2026-09-13 15:25:25--  https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz
Resolving www.robots.ox.ac.uk (www.robots.ox.ac.uk)... 129.67.94.2
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://thor.robots.ox.ac.uk/dtd/dtd-r1.0.1.tar.gz [following]
--2026-09-13 15:25:26--  https://thor.robots.ox.ac.uk/dtd/dtd-r1.0.1.tar.gz
Resolving thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)... 129.67.95.98
Connecting to thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)|129.67.95.98|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 625239812 (596M) [application/octet-stream]
Saving to: ‘/kaggle/working/dtd-r1.0.1.tar.gz’

/kaggle/working/dtd 100%[===================>] 596.27M  19.8MB/s    in 37s     

2026-09-13 15:26:03 (16.2 MB/s) - ‘/kaggle/working/dtd-r1.0.1.tar.gz’ saved [625239812/625239812]

dtd/
dtd/labels/
dtd/labels/test2.txt
dtd/labels/train

  0%|          | 0/23 [00:00<?, ?it/s]

Textures feature shape: torch.Size([5640, 512])
Feature norm: tensor(1.0000)

Mean ID score       : -0.25042668
Mean Textures score : -0.4174507

========== TEXTURES OOD RESULTS ==========
AUROC : 89.47
AUPR  : 94.02
FPR95 : 50.99


In [29]:
# ============================================================
# Download Places365 test set used by original NPOS
# ============================================================

!mkdir -p /kaggle/working/places365_test

!wget -O /kaggle/working/test_256.tar \
http://data.csail.mit.edu/places/places365/test_256.tar

!tar -xf /kaggle/working/test_256.tar \
-C /kaggle/working/places365_test

# ============================================================
# Places365 OOD Evaluation — official test_256 / NPOS-style
# ============================================================

import os
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Official Places365 test set path
# ------------------------------------------------------------

PLACES_ROOT = "/kaggle/working/places365_test/test_256"


# ------------------------------------------------------------
# 2. Original NPOS Places transform
# ------------------------------------------------------------

places_transform = transforms.Compose([
    transforms.Resize(32),
    transforms.CenterCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5071, 0.4867, 0.4408),
        (0.2675, 0.2565, 0.2761)
    ),
])


# ------------------------------------------------------------
# 3. Flat image dataset
# ------------------------------------------------------------

class FlatImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        self.files = sorted([
            f for f in os.listdir(root)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.files[idx])

        image = Image.open(path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, -1


places_dataset = FlatImageDataset(
    PLACES_ROOT,
    transform=places_transform
)

print("Original Places samples:", len(places_dataset))


# ------------------------------------------------------------
# 4. Randomly sample 10,000 images like original NPOS
# ------------------------------------------------------------

rng = np.random.default_rng(42)

indices = rng.choice(
    len(places_dataset),
    size=10000,
    replace=False
)

places_eval_dataset = Subset(
    places_dataset,
    indices
)

print("Places evaluation samples:", len(places_eval_dataset))


# ------------------------------------------------------------
# 5. DataLoader
# ------------------------------------------------------------

places_loader = DataLoader(
    places_eval_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


# ------------------------------------------------------------
# 6. Extract normalized 512-d features
# ------------------------------------------------------------

@torch.no_grad()
def extract_ood_features(model, loader, device):

    model.eval()
    features = []

    for images, _ in tqdm(loader):

        images = images.to(device)

        h = model.encoder(images)
        h = F.normalize(h, dim=1)

        features.append(h.cpu())

    return torch.cat(features, dim=0)


fplaces = extract_ood_features(
    model,
    places_loader,
    DEVICE
)

print("Places feature shape:", fplaces.shape)
print("Feature norm:", fplaces[0].norm())


# ------------------------------------------------------------
# 7. KNN scoring
# ------------------------------------------------------------

K = 300

D_places, _ = index.search(
    fplaces.numpy(),
    K
)

scores_places = -D_places[:, -1]

print("\nMean ID score     :", scores_in.mean())
print("Mean Places score :", scores_places.mean())


# ------------------------------------------------------------
# 8. OOD metrics
# ------------------------------------------------------------

auroc_places, aupr_places, fpr95_places = get_measures(
    scores_in,
    scores_places
)

print("\n========== PLACES365 OOD RESULTS ==========")
print(f"AUROC : {auroc_places * 100:.2f}")
print(f"AUPR  : {aupr_places * 100:.2f}")
print(f"FPR95 : {fpr95_places * 100:.2f}")
print("===========================================")

--2026-09-13 15:34:56--  http://data.csail.mit.edu/places/places365/test_256.tar
Resolving data.csail.mit.edu (data.csail.mit.edu)... 128.52.131.233
Connecting to data.csail.mit.edu (data.csail.mit.edu)|128.52.131.233|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://data.csail.mit.edu/places/places365/test_256.tar [following]
--2026-09-13 15:34:57--  https://data.csail.mit.edu/places/places365/test_256.tar
Connecting to data.csail.mit.edu (data.csail.mit.edu)|128.52.131.233|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4736829440 (4.4G) [application/x-tar]
Saving to: ‘/kaggle/working/test_256.tar’

/kaggle/working/tes 100%[===================>]   4.41G  67.5MB/s    in 68s     

2026-09-13 15:36:05 (66.0 MB/s) - ‘/kaggle/working/test_256.tar’ saved [4736829440/4736829440]

Original Places samples: 328500
Places evaluation samples: 10000


  0%|          | 0/40 [00:00<?, ?it/s]

Places feature shape: torch.Size([10000, 512])
Feature norm: tensor(1.)

Mean ID score     : -0.25042668
Mean Places score : -0.32838926

========== PLACES365 OOD RESULTS ==========
AUROC : 74.35
AUPR  : 73.76
FPR95 : 76.98


In [18]:
# ============================================================
# LSUN-C OOD Evaluation — CIFAR-100 / NPOS-style
# ============================================================

# Download + extract LSUN-C
!wget -O /kaggle/working/LSUN.tar.gz \
https://www.dropbox.com/s/fhtsw1m3qxlwj6h/LSUN.tar.gz

!tar -xvzf /kaggle/working/LSUN.tar.gz \
-C /kaggle/working/


import os
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. LSUN-C path
# ------------------------------------------------------------

LSUN_C_ROOT = "/kaggle/working/LSUN/test"


# ------------------------------------------------------------
# 2. LSUN-C transform
# Downloaded archive contains 36x36 images.
# Resize to 32x32 to match CIFAR-100 input resolution.
# ------------------------------------------------------------

lsun_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5071, 0.4867, 0.4408),
        (0.2675, 0.2565, 0.2761)
    ),
])


# ------------------------------------------------------------
# 3. Custom dataset
# Images are directly inside /LSUN/test
# ------------------------------------------------------------

class FlatImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        self.files = sorted([
            f for f in os.listdir(root)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.files[idx])

        image = Image.open(path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        # Dummy label — not used for OOD evaluation
        return image, -1


lsun_dataset = FlatImageDataset(
    LSUN_C_ROOT,
    transform=lsun_transform
)

print("LSUN-C samples:", len(lsun_dataset))


# ------------------------------------------------------------
# 4. DataLoader
# ------------------------------------------------------------

lsun_loader = DataLoader(
    lsun_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


# ------------------------------------------------------------
# 5. Extract normalized 512-d features
# ------------------------------------------------------------

@torch.no_grad()
def extract_ood_features(model, loader, device):

    model.eval()
    features = []

    for images, _ in tqdm(loader):

        images = images.to(device)

        # 512-d penultimate representation
        h = model.encoder(images)

        # Match original NPOS KNN evaluation
        h = F.normalize(h, dim=1)

        features.append(h.cpu())

    return torch.cat(features, dim=0)


flsun = extract_ood_features(
    model,
    lsun_loader,
    DEVICE
)

print("LSUN-C feature shape:", flsun.shape)
print("Feature norm:", flsun[0].norm())


# ------------------------------------------------------------
# 6. KNN scoring
# Make sure K = 300 for CIFAR-100 test script
# ------------------------------------------------------------

K = 300

D_lsun, _ = index.search(
    flsun.numpy(),
    K
)

scores_lsun = -D_lsun[:, -1]


print("\nMean ID score     :", scores_in.mean())
print("Mean LSUN-C score :", scores_lsun.mean())


# ------------------------------------------------------------
# 7. Original-style NPOS metrics
# ------------------------------------------------------------

auroc_lsun, aupr_lsun, fpr95_lsun = get_measures(
    scores_in,
    scores_lsun
)


print("\n========== LSUN-C OOD RESULTS ==========")
print(f"AUROC : {auroc_lsun * 100:.2f}")
print(f"AUPR  : {aupr_lsun * 100:.2f}")
print(f"FPR95 : {fpr95_lsun * 100:.2f}")
print("========================================")

--2026-09-13 15:11:07--  https://www.dropbox.com/s/fhtsw1m3qxlwj6h/LSUN.tar.gz
Resolving www.dropbox.com (www.dropbox.com)... 162.125.3.18, 2620:100:6018:18::a27d:312
Connecting to www.dropbox.com (www.dropbox.com)|162.125.3.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/ohqceel2yrxuhntg0mirg/LSUN.tar.gz?rlkey=l2ovcmekq2gj529m3b2hw2ppp [following]
--2026-09-13 15:11:07--  https://www.dropbox.com/scl/fi/ohqceel2yrxuhntg0mirg/LSUN.tar.gz?rlkey=l2ovcmekq2gj529m3b2hw2ppp
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc6f8bfa584ddcc8c1f9be65c676.dl.dropboxusercontent.com/cd/0/inline/DID3bDm2nEjuhJfRiV0umJGw6FoUo7T8BpztHhy7-NfOqA8ZMNLZdmKfVapUNU_8_qJ4Ndeft2RfAceTYjSjyisliVAefJSW85pnbtpeWC0E87TyLMpKELgt9DHyq-KYCO13BLm4PPFZWNhiuml3Pnj3/file# [following]
--2026-09-13 15:11:08--  https://uc6f8bfa584ddcc8c1f9be65c676.dl.dropboxusercontent.com/cd/0/inline/DID3

  0%|          | 0/40 [00:00<?, ?it/s]

LSUN-C feature shape: torch.Size([10000, 512])
Feature norm: tensor(1.)

Mean ID score     : -0.25042668
Mean LSUN-C score : -0.47674805

========== LSUN-C OOD RESULTS ==========
AUROC : 92.52
AUPR  : 92.52
FPR95 : 33.37


In [23]:
!wget -O /kaggle/working/iSUN.tar.gz \
https://www.dropbox.com/s/ssz7qxfqae0cca5/iSUN.tar.gz

!tar -xvzf /kaggle/working/iSUN.tar.gz \
-C /kaggle/working/


# ============================================================
# iSUN OOD Evaluation — CIFAR-100 / NPOS-style
# ============================================================

import os
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. iSUN path
# ------------------------------------------------------------

ISUN_ROOT = "/kaggle/working/iSUN/iSUN_patches"


# ------------------------------------------------------------
# 2. Original NPOS iSUN transform
# Original repo uses Resize(32), ToTensor(), normalize
# Images here are already 32x32, so this leaves size unchanged
# ------------------------------------------------------------

isun_transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5071, 0.4867, 0.4408),
        (0.2675, 0.2565, 0.2761)
    ),
])


# ------------------------------------------------------------
# 3. Flat image dataset
# ------------------------------------------------------------

class FlatImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        self.files = sorted([
            f for f in os.listdir(root)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.files[idx])

        image = Image.open(path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, -1


isun_dataset = FlatImageDataset(
    ISUN_ROOT,
    transform=isun_transform
)

print("iSUN samples:", len(isun_dataset))


# ------------------------------------------------------------
# 4. DataLoader
# ------------------------------------------------------------

isun_loader = DataLoader(
    isun_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


# ------------------------------------------------------------
# 5. Extract normalized 512-d features
# ------------------------------------------------------------

@torch.no_grad()
def extract_ood_features(model, loader, device):

    model.eval()
    features = []

    for images, _ in tqdm(loader):

        images = images.to(device)

        h = model.encoder(images)
        h = F.normalize(h, dim=1)

        features.append(h.cpu())

    return torch.cat(features, dim=0)


fisun = extract_ood_features(
    model,
    isun_loader,
    DEVICE
)

print("iSUN feature shape:", fisun.shape)
print("Feature norm:", fisun[0].norm())


# ------------------------------------------------------------
# 6. KNN scoring
# ------------------------------------------------------------

K = 300

D_isun, _ = index.search(
    fisun.numpy(),
    K
)

scores_isun = -D_isun[:, -1]

print("\nMean ID score   :", scores_in.mean())
print("Mean iSUN score :", scores_isun.mean())


# ------------------------------------------------------------
# 7. Original-style NPOS metrics
# ------------------------------------------------------------

auroc_isun, aupr_isun, fpr95_isun = get_measures(
    scores_in,
    scores_isun
)

print("\n========== iSUN OOD RESULTS ==========")
print(f"AUROC : {auroc_isun * 100:.2f}")
print(f"AUPR  : {aupr_isun * 100:.2f}")
print(f"FPR95 : {fpr95_isun * 100:.2f}")
print("======================================")

--2026-09-13 15:14:49--  https://www.dropbox.com/s/ssz7qxfqae0cca5/iSUN.tar.gz
Resolving www.dropbox.com (www.dropbox.com)... 162.125.3.18, 2620:100:6018:18::a27d:312
Connecting to www.dropbox.com (www.dropbox.com)|162.125.3.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/wpkzixs1zbqomg5ufq0dd/iSUN.tar.gz?rlkey=46mty3ly8kk3vdxtlnmdjc6zu [following]
--2026-09-13 15:14:50--  https://www.dropbox.com/scl/fi/wpkzixs1zbqomg5ufq0dd/iSUN.tar.gz?rlkey=46mty3ly8kk3vdxtlnmdjc6zu
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc4460864b644f9bde742b41937b.dl.dropboxusercontent.com/cd/0/inline/DIAsiPzVNrKZ8FZtJchKH0yBPovHVxXJi8lWHppsc_xH_GBWIDahs5EXEBs6KTl5DhSA2iAENUkt7dIf3qwzbRQq-fu3DOAOoF_c1lxx9VTkGS_YDCv12oTtRq39pBe05tCu6Th9h3beDhE9xt72d6qr/file# [following]
--2026-09-13 15:14:50--  https://uc4460864b644f9bde742b41937b.dl.dropboxusercontent.com/cd/0/inline/DIAs

  0%|          | 0/35 [00:00<?, ?it/s]

iSUN feature shape: torch.Size([8925, 512])
Feature norm: tensor(1.0000)

Mean ID score   : -0.25042668
Mean iSUN score : -0.41603467

========== iSUN OOD RESULTS ==========
AUROC : 90.73
AUPR  : 91.92
FPR95 : 42.94
